# ReservoirLSTM — Train Theo Nhánh Sông (Vu Gia & Thu Bồn) & Train Từng Hồ
**Kiến trúc**: Hindcast Bi-LSTM + Cross-Attention + NWP Embedding + Quantile Head (7 mức)

Notebook này tự động phân loại 16 hồ thủy điện thành 2 Nhánh Sông chính:
- **Nhánh Sông Vu Gia (11 hồ)**: `HO DAK MI 2`, `HO DAK MI 3`, `HO DAK MI 4`, `HO SONG BUNG 2`, `HO SONG BUNG 4`, `HO SONG BUNG 4A`, `HO SONG BUNG 5`, `HO SONG BUNG 6`, `HO A VUONG`, `HO SONG CON 2`, `HO ZA HUNG`.
- **Nhánh Sông Thu Bồn (5 hồ)**: `HO SONG TRANH 2`, `HO SONG TRANH 3`, `HO SONG TRANH 4`, `HO KHE DIEN`, `HO DAK MI 4C`.

**Các bước thực hiện**:
1. **Pha 1**: Train 2 model riêng biệt theo 2 nhánh sông (Model Vu Gia & Model Thu Bồn).
2. **Pha 2**: Đánh giá chỉ số của model Nhánh sông trên tập test từng hồ.
3. **Pha 3**: Train model độc lập cho từng hồ riêng lẻ (Single Reservoir Model).
4. **Pha 4**: Tổng hợp và xuất **Bảng so sánh chỉ số NSE, RMSE, MAE, R²** khớp đúng định dạng bảng yêu cầu.


In [ ]:
import os, json, math, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, ConcatDataset
from tqdm import tqdm
warnings.filterwarnings("ignore")

OUTPUT_ROOT = "/kaggle/working"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Tự động quét /kaggle/input hoặc tải từ Hugging Face (Anvo2004/dataset_all_lake)
HF_REPO_ID = "Anvo2004/dataset_all_lake"
RESERVOIR_DATA_DIRS = {}

for _root, _dirs, _files in os.walk("/kaggle/input"):
    if "v2_X_hindcast.npy" in _files:
        RESERVOIR_DATA_DIRS[os.path.basename(_root)] = _root

if not RESERVOIR_DATA_DIRS:
    print(f"Không tìm thấy data ở /kaggle/input -> Đang tự động tải từ Hugging Face Dataset: '{HF_REPO_ID}'...")
    try:
        from huggingface_hub import hf_hub_download
        import zipfile
        zip_path = hf_hub_download(repo_id=HF_REPO_ID, filename="datasets_all_reservoirs.zip", repo_type="dataset")
        print(f"Tải thành công từ Hugging Face: {zip_path}, đang giải nén...")
        extract_dir = f"{OUTPUT_ROOT}/datasets"
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        for _root, _dirs, _files in os.walk(extract_dir):
            if "v2_X_hindcast.npy" in _files:
                RESERVOIR_DATA_DIRS[os.path.basename(_root)] = _root
    except Exception as e:
        print(f"Lưu ý Hugging Face: {e}. Vui lòng đảm bảo file 'datasets_all_reservoirs.zip' đã được upload lên repo '{HF_REPO_ID}'.")

print(f"Tìm thấy {len(RESERVOIR_DATA_DIRS)} thư mục dữ liệu hồ:")
for k, v in sorted(RESERVOIR_DATA_DIRS.items()):
    print(f"  {k} -> {v}")


## (Tuy chon) Build lai dataset co Station Rain Attention tren Kaggle (Kich ban 2)

Neu ban da attach 2 Kaggle Dataset: (1) code nguon `LSTM_Py_Backend_v2` (cac thu muc `config/`, `data/`, `models/`, `training/`, `features/` -- chi ~150KB, khong phai dataset da build), va (2) thu muc Excel goc `Data_Tung_Ho_Ma_Tran_Rong` (~22MB) -- cell duoi day se TU DONG build lai toan bo 16 ho (bao gom `v2_station_rain.npy`/`v2_station_mask.npy` cho StationRainAttention) ngay tren Kaggle, khong can transfer ~29GB dataset da build qua tay. Mat khoang 20-25 phut (can bat Internet cho Kaggle notebook de goi Open-Meteo).

Neu chua muon dung Station Attention, dat `ENABLE_STATION_ATTENTION_REBUILD = False` hoac bo qua cell nay -- notebook van chay binh thuong voi du lieu da co o `RESERVOIR_DATA_DIRS` (cell tren).

In [ ]:
ENABLE_STATION_ATTENTION_REBUILD = True  # doi thanh False de bo qua buoc nay

if ENABLE_STATION_ATTENTION_REBUILD:
    import glob

    code_candidates = glob.glob("/kaggle/input/*/config") + glob.glob("/kaggle/input/*/*/config")
    excel_candidates = (
        glob.glob("/kaggle/input/*/Data_Tung_Ho_Ma_Tran_Rong")
        + glob.glob("/kaggle/input/*/*/Data_Tung_Ho_Ma_Tran_Rong")
    )

    if not code_candidates:
        print(
            "[BO QUA Station Attention] Khong tim thay code nguon LSTM_Py_Backend_v2 "
            "(thu muc config/) trong /kaggle/input -- hay attach Kaggle Dataset chua "
            "config/, data/, models/, training/, features/. Notebook se tiep tuc voi "
            "du lieu hien co o RESERVOIR_DATA_DIRS (khong co Station Attention)."
        )
    elif not excel_candidates:
        print(
            "[BO QUA Station Attention] Khong tim thay Data_Tung_Ho_Ma_Tran_Rong trong "
            "/kaggle/input -- hay attach Kaggle Dataset chua thu muc Excel goc. Notebook "
            "se tiep tuc voi du lieu hien co o RESERVOIR_DATA_DIRS (khong co Station Attention)."
        )
    else:
        CODE_ROOT = os.path.dirname(code_candidates[0])
        EXCEL_ROOT = os.path.dirname(excel_candidates[0])
        print(f"Tim thay code nguon: {CODE_ROOT}")
        print(f"Tim thay Excel goc: {EXCEL_ROOT}")

        sys.path.insert(0, CODE_ROOT)

        from config.reservoirs import RESERVOIRS as _RESERVOIRS_FOR_BUILD
        from config.settings import ReservoirLSTMConfig as _CfgForBuild
        from data.dataset_builder import build_reservoir_dataset

        STATION_DATASETS_ROOT = f"{OUTPUT_ROOT}/datasets_station"
        os.makedirs(STATION_DATASETS_ROOT, exist_ok=True)

        print("=" * 70)
        print(f"BUILD LAI {len(_RESERVOIRS_FOR_BUILD)} HO VOI STATION RAIN ATTENTION (~20-25 phut)")
        print("=" * 70)

        _orig_cwd = os.getcwd()
        try:
            # rain_matrix_loader tim Data_Tung_Ho_Ma_Tran_Rong/<ten_ho>/ theo duong dan
            # TUONG DOI voi cwd -- chuyen cwd tam thoi ve thu muc chua no.
            os.chdir(EXCEL_ROOT)
            for rid, info in _RESERVOIRS_FOR_BUILD.items():
                cfg_build = _CfgForBuild(rid=rid, reservoir_name=info["name"])
                cfg_build.use_station_attention = True
                try:
                    build_reservoir_dataset(
                        rid=rid, start_date="2022-01-01", end_date="2025-12-31",
                        cfg=cfg_build, out_root=STATION_DATASETS_ROOT, fetch_nwp=True,
                    )
                except Exception as e:
                    print(f"[SKIP] rid={rid} {info['name']}: {e}")
        finally:
            os.chdir(_orig_cwd)

        # Ghi de RESERVOIR_DATA_DIRS de cac Pha sau dung du lieu vua build (co station rain)
        n_updated = 0
        for rid, info in _RESERVOIRS_FOR_BUILD.items():
            key = info["name"].replace(" ", "_")
            d = os.path.join(STATION_DATASETS_ROOT, key)
            if os.path.exists(d):
                RESERVOIR_DATA_DIRS[key] = d
                n_updated += 1
        print()
        print(f"Da cap nhat RESERVOIR_DATA_DIRS -> {n_updated}/{len(_RESERVOIRS_FOR_BUILD)} ho tro vao {STATION_DATASETS_ROOT}")


## Config (config/reservoirs.py + config/settings.py)

In [ ]:
# config/reservoirs.py
# Copy nguyên từ LSTM_Py_Backend/lstm_service/config/reservoirs.py — 16 hồ Quảng Nam/Đà Nẵng.
# "idx" giữ lại để tra cứu INFLOW_CAPS_M3S/RESERVOIR_TO_STATIONS (không dùng làm reservoir
# embedding nữa — LSTM_Py_Backend_v2 train 1 model độc lập cho từng "rid").

RESERVOIRS = {
    1: {"idx": 0, "name": "HO A VUONG", "lat": 15.815, "lon": 107.63, "river_basin": "Vu Gia"},
    2: {"idx": 1, "name": "HO DAK MI 4", "lat": 15.45285, "lon": 107.83250, "river_basin": "Vu Gia"},
    3: {"idx": 2, "name": "HO SONG BUNG 4", "lat": 15.726, "lon": 107.637, "river_basin": "Vu Gia"},
    4: {"idx": 3, "name": "HO SONG TRANH 2", "lat": 15.326, "lon": 108.125, "river_basin": "Thu Bồn"},
    7: {"idx": 4, "name": "HO SONG BUNG 4A", "lat": 15.765, "lon": 107.679, "river_basin": "Vu Gia"},
    8: {"idx": 5, "name": "HO SONG BUNG 5", "lat": 15.808, "lon": 107.7473, "river_basin": "Vu Gia"},
    9: {"idx": 6, "name": "HO SONG BUNG 2", "lat": 15.7145, "lon": 107.3970, "river_basin": "Vu Gia"},
    11: {"idx": 7, "name": "HO SONG BUNG 6", "lat": 15.82, "lon": 107.78, "river_basin": "Vu Gia"},
    12: {"idx": 8, "name": "HO SONG TRANH 3", "lat": 15.4445, "lon": 108.1430, "river_basin": "Thu Bồn"},
    13: {"idx": 9, "name": "HO ZA HUNG", "lat": 15.86005, "lon": 107.654, "river_basin": "Vu Gia"},
    14: {"idx": 10, "name": "HO DAK MI 3", "lat": 15.33, "lon": 107.81, "river_basin": "Vu Gia"},
    15: {"idx": 11, "name": "HO KHE DIEN", "lat": 15.71279, "lon": 107.92872, "river_basin": "Thu Bồn"},
    16: {"idx": 12, "name": "HO SONG CON 2", "lat": 15.90558, "lon": 107.8234, "river_basin": "Vu Gia"},
    17: {"idx": 13, "name": "HO SONG TRANH 4", "lat": 15.53666, "lon": 108.152, "river_basin": "Thu Bồn"},
    18: {"idx": 14, "name": "HO DAK MI 2", "lat": 15.23832, "lon": 107.8100, "river_basin": "Vu Gia"},
    19: {"idx": 15, "name": "HO DAK MI 4C", "lat": 15.4643, "lon": 107.92893, "river_basin": "Thu Bồn"},
}

VU_GIA_RIDS = [rid for rid, info in RESERVOIRS.items() if info["river_basin"] == "Vu Gia"]
THU_BON_RIDS = [rid for rid, info in RESERVOIRS.items() if info["river_basin"] == "Thu Bồn"]

RIVER_BASINS = {
    "Vu Gia": VU_GIA_RIDS,
    "Thu Bồn": THU_BON_RIDS,
}


# Kich ban 4 (nguoi dung de xuat): cac ho ha luu nhanh Song Bung chiu anh
# huong truc tiep tu luu luong xa cua ho thuong nguon. Day la NEN TANG
# config cho tinh nang "them Q_outflow ho thuong nguon lam feature dau vao"
# -- CHUA duoc noi vao dataset_builder.py/model (can rebuild dataset + doi
# input dim). Xem ghi chu Markdown 'Pha 4.5' trong train_all_reservoirs.ipynb.
UPSTREAM_RESERVOIRS = {
    7:  [9, 3, 18],   # Song Bung 4A  <- Song Bung 2, Song Bung 4, Dak Mi 2
    8:  [9, 3, 18],   # Song Bung 5   <- Song Bung 2, Song Bung 4, Dak Mi 2
    11: [9, 3, 18],   # Song Bung 6   <- Song Bung 2, Song Bung 4, Dak Mi 2
}


def get_reservoirs_by_basin(basin_name: str) -> dict:
    """Trả về dict chứa các hồ thuộc lưu vực sông chỉ định."""
    return {rid: info for rid, info in RESERVOIRS.items() if info["river_basin"] == basin_name}


In [ ]:
"""
Config cho LSTM_Py_Backend_v2 — 1 model độc lập cho MỖI hồ (không reservoir
embedding, không train chung 16 hồ).

Lý do tách khỏi lstm_service/config/settings_v2.py (FloodLSTMv2Config):
  - Baseline NSE cũ (artifacts/plots/metrics.txt, lstm_service) cho thấy model
    chung có độ lệch rất lớn giữa các hồ (0.87 xuống tới âm) — nghi ngờ 1 vài
    hồ dữ liệu xấu (xem INFLOW_CAPS_M3S bên dưới, đánh dấu "lỗi data") kéo NSE
    trung bình xuống hoặc gây nhiễu học chung.
  - Train riêng từng hồ: đổi 1 hồ dữ liệu xấu KHÔNG ảnh hưởng tới model của
    các hồ khác, dễ debug/so sánh NSE per-reservoir hơn.

Mặc định hindcast/forecast GIỮ NGUYÊN theo hợp đồng production hiện tại
(SEQ_LENGTH=240h/10 ngày, HORIZON=24h — xem lstm_service/config/settings.py)
để model mới có thể thay thế v1 sau này mà không cần đổi phía Node.js/cron.
Có thể chỉnh forecast_len=168 (7 ngày, kiểu Google FloodHub) nếu muốn.
"""

from dataclasses import dataclass, field


@dataclass
class ReservoirLSTMConfig:
    # ── Hồ đang train (bắt buộc set trước khi build dataset/train) ─────────────
    rid: int = 0                     # key trong config/reservoirs.py RESERVOIRS
    reservoir_name: str = ""         # điền tự động từ RESERVOIRS[rid]["name"]

    # ── Sequence lengths (giữ theo hợp đồng production v1) ──────────────────────
    hindcast_len: int = 240          # 10 ngày lịch sử (hourly) — SEQ_LENGTH cũ
    forecast_len: int = 24           # 24h dự báo — HORIZON cũ
    #   Muốn thử 7-ngày kiểu FloodLSTMv2/Google: hindcast_len=720, forecast_len=168

    # ── Model dimensions ──────────────────────────────────────────────────────
    hidden_size: int = 128           # nhỏ hơn bản dùng chung (256) vì data/model giờ nhỏ hơn nhiều
    num_layers: int = 2
    nwp_embed_dim: int = 32

    # ── Input features ─────────────────────────────────────────────────────────
    # 47 = 18 rain + 12 inflow + 6 reservoir + 5 meteo (temp/rh/pressure/et0/wind,
    # đủ 5 vì giờ có Open-Meteo archive — xem data/nwp_fetcher.py) + 6 thời gian
    n_hindcast_features: int = 47    # xem data/dataset_builder.py FEATURES
    n_nwp_features: int = 6          # rain_fc, rain_fc_3h, rain_fc_6h, rain_fc_24h, temp_fc, wind_fc
    n_nwp_sources: int = 1           # chỉ Open-Meteo — không có nguồn dự phòng như bản chung

    # ── Học trọng số trạm mưa (tùy chọn, TẮT mặc định) ───────────────────────────
    # data/rain_matrix_loader.py::load_station_rain_matrix() đã parse thật mưa từng
    # trạm từ Data_Tung_Ho_Ma_Tran_Rong/*.xlsx (Col 121+i*24, đã verify trên file
    # thật — xem data/dataset_builder.py). Bật cờ này để dataset_builder.py build
    # v2_station_rain.npy/v2_station_mask.npy/v2_station_prior_weights.npy — mặc
    # định TẮT vì cần rebuild dataset + train lại từ đầu (đổi input dim của model).
    use_station_attention: bool = False
    max_stations: int = 7            # số trạm tối đa/hồ, xem RESERVOIR_TO_STATIONS

    # ── Khử điểm nhiễu/outlier (Hampel filter) ───────────────────────────────────
    # Bổ sung cho INFLOW_CAPS_M3S (chỉ chặn trần cứng): rolling median + MAD có thể
    # phát hiện các điểm lệch bất thường nằm DƯỚI cap (vd cảm biến nhiễu ngắn hạn) —
    # xem data/dataset_builder.py::_hampel_despike(). ĐÃ TEST trên dữ liệu thật
    # (datasets/*/v2_y.npy): với window=7, tỷ lệ điểm bị sửa KHÔNG đặc hiệu cho các
    # hồ "lỗi data" — Sông Tranh 2 (hồ tốt, NSE=0.496) bị đụng ~4-5% điểm trong khi
    # Đắk Mi 4C/Sông Bung 6 (2 hồ nghi lỗi data nhất) chỉ ~0.02-0.16%. Tức là filter
    # generic nhạy với biến động giờ-theo-giờ tự nhiên của hồ lưu lượng lớn hơn là
    # nhạy với lỗi cảm biến thật. MẶC ĐỊNH TẮT — chỉ bật thủ công (per-reservoir)
    # sau khi build thử và xem log "Despike: thay N điểm" in ra; > ~1% nên coi là
    # dấu hiệu cần chỉnh window/n_sigmas riêng cho hồ đó, không dùng nguyên default.
    despike: bool = False
    despike_window: int = 7          # số giờ trong cửa sổ rolling median (lẻ)
    despike_n_sigmas: float = 8.0    # ngưỡng lệch (x MAD quy đổi độ lệch chuẩn) để coi là outlier

    # ── Quantile output (7 mức, giống FloodLSTMv2) ───────────────────────────────
    quantiles: list = field(
        default_factory=lambda: [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    )

    # ── Training hyperparameters ────────────────────────────────────────────────
    batch_size: int = 128
    epochs: int = 100
    lr: float = 3e-4
    weight_decay: float = 1e-3
    warmup_epochs: int = 10
    grad_clip: float = 1.0
    # 20 -> 30: tránh dừng sớm khi val loss còn dao động (đặc biệt val set nhỏ
    # của 1 hồ, ~2-3K mẫu) trước khi thực sự hội tụ — xem chẩn đoán NSE thấp.
    patience: int = 30

    teacher_forcing_start: float = 0.8
    teacher_forcing_end: float = 0.0

    oversample_p95_factor: int = 2
    # 3 -> 6: cung ly do peak_weight o tren -- nhan ban manh hon top 1%%
    # gia tri cuc tri de model thay nhieu mau lu that hon trong 1 epoch.
    oversample_p99_factor: int = 6
    # Bo sung oversample theo % cuc tri o tren (chi bat DIEM DINH rieng le):
    # nhan ban them CA giai doan mua mua (khong chi diem dinh) de model hoc ky
    # hon dang tang/giam cua tran lu, khong chi hoc gia tri tai 1 thoi diem don
    # le -- xem train_reservoir.py/pretrain_pooled.py phan "Oversampling mua".
    rainy_season_months: list = field(default_factory=lambda: [8, 9, 10, 11, 12])
    oversample_rainy_season_factor: int = 1

    # ── Loss weights (quantile_loss_v2) ─────────────────────────────────────────
    # Trước đây hard-code trong models/quantile_loss_v2.py, giờ đưa vào config để
    # tinh chỉnh/ablation được (vd peak_weight=0, coverage_weight=0 -> pinball
    # loss thuần, so sánh NSE để tách bạch "model dở" khỏi "loss không tối ưu
    # trực tiếp NSE" — xem chẩn đoán NSE thấp per-reservoir).
    horizon_decay: float = 0.02
    coverage_weight: float = 0.05
    # 0.15 -> 0.35: du bao dinh lu hut 50-80% so voi thuc te (xem chan doan
    # event_diagnostics tren ket qua train that) -- tang manh trong so MSE
    # rieng cho P50 tai cac diem lu de buoc model uu tien do chinh xac dinh
    # hon la phan phoi quantile ho quan.
    peak_weight: float = 0.35

    target_noise_std: float = 0.005

    # ── Training splits (fixed-date, giữ theo train_global.py) ─────────────────
    train_end: str = "2024-08-31"
    val_start: str = "2024-09-01"
    val_end: str = "2025-01-01"
    test_start: str = "2025-09-01"

    # ── Paths ──────────────────────────────────────────────────────────────────
    data_dir: str = "."
    artifacts_dir: str = "artifacts"

    @property
    def n_quantiles(self) -> int:
        return len(self.quantiles)

    @property
    def median_idx(self) -> int:
        return self.n_quantiles // 2

    def teacher_forcing_ratio(self, epoch: int) -> float:
        p = epoch / max(self.epochs - 1, 1)
        return self.teacher_forcing_start * (1 - p) + self.teacher_forcing_end * p


# ═══════════════════════════════════════════════════════════════════════════════
# Giới hạn inflow hợp lý (m3/s) — copy từ lstm_service/data/dataset_builder.py.
# Các hồ đánh dấu "lỗi data" là nghi phạm hàng đầu cho NSE thấp trong baseline
# cũ (Song Bung 2 idx6, Song Bung 6 idx7, Dak Mi 3 idx10, Khe Dien idx11,
# Dak Mi 2 idx14) — ưu tiên kiểm tra lại coverage/outlier khi train hồ này.
# ═══════════════════════════════════════════════════════════════════════════════
INFLOW_CAPS_M3S = {
    1:  2500,   # HO A VUONG        (idx=0)
    2:  4500,   # HO DAK MI 4       (idx=1)
    3:  4500,   # HO SONG BUNG 4    (idx=2)
    4:  8000,   # HO SONG TRANH 2   (idx=3)
    7:  7000,   # HO SONG BUNG 4A   (idx=4)
    8:  7000,   # HO SONG BUNG 5    (idx=5)
    9:  800,    # HO SONG BUNG 2    (idx=6)  — lỗi data (p99.9=617, max quan trắc=19701)
    11: 8000,   # HO SONG BUNG 6    (idx=7)  — lỗi data (p99.9=5946, max quan trắc=40232)
    12: 12000,  # HO SONG TRANH 3   (idx=8)
    13: 2500,   # HO ZA HUNG        (idx=9)
    14: 2000,   # HO DAK MI 3       (idx=10) — lỗi data (p99.9=1350, max quan trắc=35321)
    15: 1000,   # HO KHE DIEN       (idx=11) — lỗi data (p99.9=729,  max quan trắc=6087)
    16: 900,    # HO SONG CON 2     (idx=12)
    17: 13000,  # HO SONG TRANH 4   (idx=13)
    18: 2500,   # HO DAK MI 2       (idx=14) — lỗi data rõ ràng (p99.9=1537, max quan trắc=391526)
    19: 700,    # HO DAK MI 4C      (idx=15)
}


## Model — StationRainAttention

In [ ]:
# models/station_attention.py
"""
StationRainAttention — bản 1-hồ (không có reservoir embedding/context).

Khác bản ở lstm_service/models/station_attention.py: bản đó phục vụ 1 model
CHUNG nhiều hồ nên cần bảng bias theo (reservoir, station) + reservoir embedding
làm context cho score_net. Ở đây mỗi model chỉ phục vụ ĐÚNG 1 hồ, nên:
  - prior_bias là 1 vector duy nhất (không phải bảng tra theo hồ)
  - score_net chỉ nhận giá trị mưa hiện tại (không cần context "hồ nào")

Thiết kế attention giữ nguyên ý tưởng gốc (lấy cảm hứng từ AttenCLSTM trong
CNN-LSTM-Attention-Model-for-Runoff-Prediction): giữ IDW làm prior vật lý
(bias khởi tạo từ log(idw_weight) qua init_prior()), rồi để attention học điều
chỉnh dần trong quá trình train. Trạm thiếu dữ liệu tại thời điểm t bị loại
khỏi softmax qua mask.

TẮT mặc định (config.use_station_attention=False) — xem TODO trong
data/dataset_builder.py về việc parse mưa từng trạm từ Excel.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class StationRainAttention(nn.Module):
    """
    Input:
        station_rain : (B, T, S) — mưa thô từng trạm (mm), 0 nếu thiếu
        station_mask : (B, T, S) bool — True nếu trạm hợp lệ tại t

    Output:
        learned_rain : (B, T, 1) — mưa lưu vực đã học trọng số
        attn_weights : (B, T, S) — trọng số attention (để log/debug)
    """

    def __init__(self, max_stations: int, hidden_dim: int = 16):
        super().__init__()
        self.max_stations = max_stations

        # Bias tĩnh theo trạm — khởi tạo từ trọng số IDW (init_prior), học tiếp khi train
        self.prior_bias = nn.Parameter(torch.zeros(max_stations))

        # Logit động theo giá trị mưa hiện tại tại mỗi trạm
        self.score_net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        nn.init.xavier_uniform_(self.score_net[0].weight)
        nn.init.zeros_(self.score_net[0].bias)
        nn.init.zeros_(self.score_net[2].weight)
        nn.init.zeros_(self.score_net[2].bias)

    def init_prior(self, weights: list):
        """Khởi tạo bias từ trọng số IDW tĩnh đã normalize (sum=1), log space."""
        padded = list(weights) + [0.0] * (self.max_stations - len(weights))
        log_w = [math.log(w) if w > 1e-8 else -20.0 for w in padded[: self.max_stations]]
        with torch.no_grad():
            self.prior_bias.copy_(torch.tensor(log_w, dtype=torch.float32))

    def forward(self, station_rain: torch.Tensor, station_mask: torch.Tensor):
        B, T, S = station_rain.shape
        assert S == self.max_stations, f"expected {self.max_stations} stations, got {S}"

        bias = self.prior_bias.view(1, 1, -1).expand(B, T, -1)   # (B, T, S)
        dyn_score = self.score_net(station_rain.unsqueeze(-1)).squeeze(-1)  # (B, T, S)

        logits = bias + dyn_score
        logits = logits.masked_fill(~station_mask, float("-inf"))

        no_valid = (~station_mask).all(dim=-1, keepdim=True)      # (B, T, 1)
        safe_logits = torch.where(no_valid.expand_as(logits), torch.zeros_like(logits), logits)

        attn = F.softmax(safe_logits, dim=-1)
        attn = attn.masked_fill(no_valid.expand_as(attn), 0.0)

        learned_rain = (attn * station_rain).sum(dim=-1, keepdim=True)  # (B, T, 1)
        return learned_rain, attn


## Model — ReservoirLSTM

In [ ]:
# models/flood_lstm_v2.py
"""
ReservoirLSTM — bản 1-hồ của FloodLSTM v2 (lstm_service/models/flood_lstm_v2.py).

Khác bản gốc:
  1. KHÔNG có reservoir embedding / reservoir_idx — mỗi model chỉ phục vụ 1 hồ
     (xem config/settings.py ReservoirLSTMConfig, context ở đây được lấy hết
     từ chính dữ liệu của hồ đó, không cần phân biệt "hồ nào").
  2. NWPEmbedding thay NWPFusionLayer — bản gốc hỗ trợ nhiều nguồn NWP với
     attention theo availability mask; ở đây chỉ có 1 nguồn (Open-Meteo) nên
     đơn giản hoá thành 1 projection layer.
  3. StationRainAttention (tùy chọn) không cần reservoir context nữa — xem
     models/station_attention.py bản 1-hồ.

Giữ nguyên từ bản gốc (không phụ thuộc số hồ):
  - Hindcast Encoder (Bi-LSTM) + Forecast Decoder (LSTM autoregressive)
  - Cross-attention: decoder query -> hindcast encoder key/value
  - Horizon-aware uncertainty scaling (P50 cố định, P5/P95 giãn theo lead-time)
  - Quantile head 7 mức
"""

import torch
import torch.nn as nn
import torch.nn.functional as F




# ═══════════════════════════════════════════════════════════════════════════════
# NWP Embedding — đơn giản hoá NWPFusionLayer (bản gốc hỗ trợ multi-source)
# ═══════════════════════════════════════════════════════════════════════════════

class NWPEmbedding(nn.Module):
    """Project NWP features (rain/temp/wind...) -> embedding, 1 nguồn duy nhất."""

    def __init__(self, input_dim: int, embed_dim: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, T, input_dim) -> (B, T, embed_dim)
        return self.proj(x)


# ═══════════════════════════════════════════════════════════════════════════════
# Hindcast Cross-Attention
# ═══════════════════════════════════════════════════════════════════════════════

class HindcastCrossAttention(nn.Module):
    """
    Cross-attention: forecast decoder query -> hindcast encoder key/value.
    Cho phép decoder tập trung vào các thời điểm quan trọng trong lịch sử
    (vd: đỉnh lũ trước đó, trạng thái mưa dài hạn).
    """

    def __init__(self, hidden_size: int, n_heads: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=n_heads, dropout=0.1, batch_first=True,
        )
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, query: torch.Tensor, key_value: torch.Tensor) -> torch.Tensor:
        ctx, _ = self.attn(query, key_value, key_value)
        out = self.norm(ctx + query)
        return out.squeeze(1)  # (B, H)


# ═══════════════════════════════════════════════════════════════════════════════
# ReservoirLSTM — Main Model
# ═══════════════════════════════════════════════════════════════════════════════

class ReservoirLSTM(nn.Module):
    """
    Two-phase flood forecasting model cho 1 hồ.

    Phase 1 — Hindcast:
        Input : x_hindcast (B, hindcast_len, n_hindcast_features)
        Encoder: Bi-LSTM 2 layers -> final state -> project -> decoder init
        Encoder output: (B, hindcast_len, H) -> key/value cho cross-attention

    Phase 2 — Forecast (autoregressive):
        NWP input: (B, forecast_len, n_nwp_features) — Open-Meteo, 1 nguồn
        Decoder: LSTM 2 layers + cross-attention -> 7 quantiles mỗi bước

    Output: (B, forecast_len, 7) trong sqrt(Q) space, đã sort monotonic
    """

    def __init__(self, config):
        super().__init__()
        H  = config.hidden_size
        NQ = config.n_quantiles

        # ── Học trọng số trạm mưa (tùy chọn) ──────────────────────────────────
        self.use_station_attention = getattr(config, "use_station_attention", False)
        if self.use_station_attention:
            self.station_attn = StationRainAttention(max_stations=config.max_stations)
        station_extra = 1 if self.use_station_attention else 0

        # ── Phase 1: Hindcast Encoder ──────────────────────────────────────────
        self.hindcast_proj = nn.Sequential(
            nn.Linear(config.n_hindcast_features + station_extra, H),
            nn.LayerNorm(H),
            nn.GELU(),
        )
        self.hindcast_encoder = nn.LSTM(
            input_size=H, hidden_size=H, num_layers=config.num_layers,
            dropout=0.2 if config.num_layers > 1 else 0.0,
            bidirectional=True, batch_first=True,
        )
        self.enc_kv_proj   = nn.Linear(H * 2, H)
        self.h_state_proj  = nn.Linear(H * 2, H)
        self.c_state_proj  = nn.Linear(H * 2, H)

        # ── Phase 2a: NWP Embedding ─────────────────────────────────────────────
        self.nwp_embed = NWPEmbedding(config.n_nwp_features, config.nwp_embed_dim)

        # ── Phase 2b: Cross-Attention (decoder -> hindcast) ─────────────────────
        self.cross_attn = HindcastCrossAttention(H, n_heads=4)

        # ── Phase 2c: Forecast Decoder ───────────────────────────────────────────
        dec_input_dim = config.nwp_embed_dim + NQ + H
        self.forecast_decoder = nn.LSTM(
            input_size=dec_input_dim, hidden_size=H, num_layers=config.num_layers,
            dropout=0.2 if config.num_layers > 1 else 0.0, batch_first=True,
        )

        # ── Output head ────────────────────────────────────────────────────────
        self.output_head = nn.Sequential(
            nn.Linear(H, H // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(H // 2, NQ),
        )

        # Horizon uncertainty scale: bước sau -> khoảng tin cậy rộng hơn (học được)
        self.horizon_unc_scale = nn.Parameter(
            torch.linspace(1.0, 2.5, config.forecast_len).unsqueeze(1)
        )  # (forecast_len, 1)

        self.config = config
        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if "weight_ih" in name or "weight_hh" in name:
                nn.init.orthogonal_(p)
            elif "bias" in name and p.dim() == 1 and "horizon_unc_scale" not in name:
                nn.init.zeros_(p)

    def _project_encoder_state(self, h_n: torch.Tensor, c_n: torch.Tensor):
        """Gộp forward+backward directions -> decoder initial state."""
        n_layers = self.config.num_layers
        h_fwd, h_bwd = h_n[0::2], h_n[1::2]
        c_fwd, c_bwd = c_n[0::2], c_n[1::2]
        h_cat = torch.cat([h_fwd, h_bwd], dim=-1)
        c_cat = torch.cat([c_fwd, c_bwd], dim=-1)
        h0 = self.h_state_proj(h_cat).contiguous()
        c0 = self.c_state_proj(c_cat).contiguous()
        return h0, c0

    def forward(
        self,
        x_hindcast: torch.Tensor,          # (B, hindcast_len, n_hindcast_features)
        x_nwp: torch.Tensor,               # (B, forecast_len, n_nwp_features)
        teacher_forcing_ratio: float = 0.0,
        y_true_sqrt: torch.Tensor = None,  # (B, forecast_len) ground truth, sqrt space
        station_rain: torch.Tensor = None, # (B, hindcast_len, max_stations)
        station_mask: torch.Tensor = None, # (B, hindcast_len, max_stations) bool
    ) -> torch.Tensor:                     # (B, forecast_len, n_quantiles)

        B = x_hindcast.size(0)
        device = x_hindcast.device
        NQ  = self.config.n_quantiles
        med = self.config.median_idx

        # ── Phase 1: Hindcast Encoding ──────────────────────────────────────────
        if self.use_station_attention and station_rain is not None:
            learned_rain, _ = self.station_attn(station_rain, station_mask)
            enc_input_raw = torch.cat([x_hindcast, learned_rain], dim=-1)
        else:
            enc_input_raw = x_hindcast
        enc_input = self.hindcast_proj(enc_input_raw)

        enc_out, (h_n, c_n) = self.hindcast_encoder(enc_input)   # enc_out: (B, T_h, 2H)
        enc_kv = self.enc_kv_proj(enc_out)                       # (B, T_h, H)
        h0, c0 = self._project_encoder_state(h_n, c_n)

        # ── Phase 2a: NWP Embedding ──────────────────────────────────────────────
        nwp_emb = self.nwp_embed(x_nwp)   # (B, forecast_len, nwp_embed_dim)

        # ── Phase 2b+c: Autoregressive Forecast ─────────────────────────────────
        outputs = []
        prev_q = torch.zeros(B, NQ, device=device)
        h_dec, c_dec = h0, c0

        for t in range(self.config.forecast_len):
            query = h_dec[-1].unsqueeze(1)          # (B, 1, H)
            ctx = self.cross_attn(query, enc_kv)     # (B, H)

            nwp_t = nwp_emb[:, t, :]                 # (B, nwp_embed_dim)
            dec_in = torch.cat([nwp_t, prev_q, ctx], dim=-1).unsqueeze(1)

            dec_out, (h_dec, c_dec) = self.forecast_decoder(dec_in, (h_dec, c_dec))
            raw_q = self.output_head(dec_out.squeeze(1))  # (B, NQ)

            scale = self.horizon_unc_scale[t].to(device)
            median_pred = raw_q[:, med:med + 1]
            scaled_q = median_pred + (raw_q - median_pred) * scale

            outputs.append(scaled_q)

            if teacher_forcing_ratio > 0.0 and y_true_sqrt is not None:
                use_gt = torch.rand(B, device=device) < teacher_forcing_ratio
                gt_q = prev_q.clone()
                gt_q[:, med] = y_true_sqrt[:, t]
                prev_q = torch.where(use_gt.unsqueeze(-1).expand_as(scaled_q), gt_q, scaled_q.detach())
            else:
                prev_q = scaled_q.detach()

        preds = torch.stack(outputs, dim=1)
        preds = torch.sort(preds, dim=-1).values
        return preds


## Loss — Quantile + Peak-aware + Coverage

In [ ]:
# models/quantile_loss_v2.py
"""
Loss function cho ReservoirLSTM — copy từ lstm_service/models/quantile_loss_v2.py
(logic không phụ thuộc số hồ, giữ nguyên).

Thành phần:
  1. Weighted Pinball Loss  — quantile regression chuẩn, 7 quantiles
  2. Peak-Aware MSE on P50  — ưu tiên đỉnh lũ, tránh mode collapse
  3. Horizon Decay          — bước gần hơn được weight cao hơn
  4. Coverage Bonus         — khuyến khích P5-P95 bao phủ thực tế
"""

import torch
import torch.nn.functional as F


def quantile_loss_v2(
    preds: torch.Tensor,        # (B, T, NQ) — sqrt space, sorted
    targets: torch.Tensor,      # (B, T)     — sqrt space
    quantiles: list,
    horizon_decay: float = 0.02,     # decay nhanh hơn bản 168h vì forecast_len ngắn (24h mặc định)
    coverage_weight: float = 0.05,
    peak_weight: float = 0.15,
) -> torch.Tensor:
    device = preds.device
    B, T, NQ = preds.shape
    med_idx = len(quantiles) // 2

    qs = torch.tensor(quantiles, dtype=torch.float32, device=device)

    # ── 1. Pinball (Quantile) Loss ─────────────────────────────────────────────
    err = targets.unsqueeze(-1) - preds
    pinball = torch.max(qs * err, (qs - 1.0) * err)

    p50_bias = torch.ones(NQ, device=device)
    p50_bias[med_idx] = 1.1
    pinball = pinball * p50_bias.view(1, 1, -1)

    # ── 2. Horizon Decay ───────────────────────────────────────────────────────
    t_weights = torch.exp(-horizon_decay * torch.arange(T, dtype=torch.float32, device=device))
    t_weights = t_weights / t_weights.sum()

    pinball_loss = (pinball * t_weights.view(1, -1, 1)).sum(dim=1).mean()

    # ── 3. Peak-Aware MSE on P50 ───────────────────────────────────────────────
    median_pred = preds[:, :, med_idx]
    peak_w = torch.sqrt(targets + 1.0)
    peak_w = peak_w / (peak_w.mean() + 1e-8)
    peak_w = peak_w.clamp(max=5.0)

    mse_peak = (peak_w * (median_pred - targets) ** 2)
    mse_peak = (mse_peak * t_weights.view(1, -1)).sum(dim=1).mean()

    # ── 4. Coverage Bonus ─────────────────────────────────────────────────────
    p_low  = preds[:, :, 0]
    p_high = preds[:, :, -1]
    below = F.relu(p_low  - targets)
    above = F.relu(targets - p_high)
    coverage_loss = (
        (below * t_weights.view(1, -1)).sum(dim=1).mean() +
        (above * t_weights.view(1, -1)).sum(dim=1).mean()
    )

    total = (
        (1.0 - peak_weight - coverage_weight) * pinball_loss
        + peak_weight * mse_peak
        + coverage_weight * coverage_loss
    )
    return total


## Metrics — NSE, RMSE, MAE, R2 + Event Diagnostics

In [ ]:
# training/event_metrics.py
"""
Copy nguyên từ lstm_service/training/event_metrics.py — logic không phụ thuộc
số hồ, dùng lại y hệt cho bản 1-hồ.

Chẩn đoán bổ sung cho đánh giá test — lấy cảm hứng từ get_peak() trong
repo tham khảo CNN-LSTM-Attention-Model-for-Runoff-Prediction
(aba-hash/CNN-LSTM-Attention-Model-for-Runoff-Prediction, data_processing.py).

  1. nse_per_horizon()      — NSE riêng cho từng nhóm lead-time.
  2. detect_flood_events() + flood_event_diagnostics()
                             — tách từng trận lũ riêng lẻ (rise -> peak -> fall)
                               thay vì NSE gộp trên top-N% timestep, tính NSE và
                               sai số đỉnh (relative error) cho TỪNG trận.
"""

import numpy as np


def nse_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """NSE (Nash-Sutcliffe Efficiency) trên 1 mảng 1D đã flatten."""
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    if ss_tot < 1e-8:
        return float("nan")
    return 1.0 - ss_res / ss_tot


def kge_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """KGE (Kling-Gupta Efficiency) tren 1 mang 1D -- bo sung cho NSE de
    tach ro loi do tuong quan (r), do lech bien thien (alpha), do lech
    trung binh (beta). Theo cach danh gia cua google-research/flood-forecasting
    (NSE + KGE), thay vi chi dung mot minh NSE."""
    obs_mean, pred_mean = obs.mean(), pred.mean()
    obs_std, pred_std = obs.std(), pred.std()
    if obs_std < 1e-8 or abs(obs_mean) < 1e-8 or pred_std < 1e-8:
        return float("nan")
    r = float(np.corrcoef(obs, pred)[0, 1])
    if np.isnan(r):
        return float("nan")
    alpha = pred_std / obs_std
    beta = pred_mean / obs_mean
    return 1.0 - float(np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))


def r2_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """Hệ số xác định R2 (Coefficient of Determination) giữa obs và pred."""
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    if ss_tot < 1e-8:
        return float("nan")
    return float(1.0 - ss_res / ss_tot)



def extract_lead_time_series(
    preds: np.ndarray,   # (N, T) point forecast, đơn vị gốc (m3/s)
    obs: np.ndarray,     # (N, T)
    lead_idx: int,       # 0-based: 0 = giờ thứ 1, 23 = giờ thứ 24, ...
):
    """
    Trích chuỗi liên tục obs/pred tại 1 lead-time cố định từ tập test dạng
    sliding-window. Giả định val/test loader shuffle=False.
    """
    return obs[:, lead_idx].copy(), preds[:, lead_idx].copy()


def nse_per_horizon(
    preds: np.ndarray,   # (N, T) point forecast (đơn vị gốc, không phải sqrt)
    obs: np.ndarray,     # (N, T)
    group_hours: int = 6,
) -> list:
    """NSE riêng cho từng nhóm lead-time (mặc định 6h/nhóm cho horizon 24h)."""
    N, T = preds.shape
    n_groups = (T + group_hours - 1) // group_hours
    results = []
    for g in range(n_groups):
        lo, hi = g * group_hours, min((g + 1) * group_hours, T)
        p = preds[:, lo:hi].reshape(-1)
        o = obs[:, lo:hi].reshape(-1)
        results.append({
            "group": g + 1,
            "hour_range": f"{lo + 1}-{hi}h",
            "nse": round(nse_single(o, p), 4),
            "n": int(p.size),
        })
    return results


def metrics_at_specific_horizons(
    preds: np.ndarray,   # (N, T) point forecast, m3/s
    obs: np.ndarray,     # (N, T)
    horizons: list = None,
) -> dict:
    """
    Tính các chỉ số NSE, RMSE, MAE, RSE (Relative Squared Error) riêng cho từng mốc thời gian:
    3h, 6h, 12h, 24h, 3d (72h), 7d (168h).
    """
    if horizons is None:
        horizons = [3, 6, 12, 24, 72, 168]

    horizon_labels = {
        3: "3h", 6: "6h", 12: "12h", 24: "24h",
        72: "3d", 168: "7d"
    }

    results = {}
    N, T = preds.shape
    for h in horizons:
        label = horizon_labels.get(h, f"{h}h")
        idx = min(h - 1, T - 1)  # 0-based index, fallback to last step if forecast_len < h
        if idx >= 0:
            o_h = obs[:, idx]
            p_h = preds[:, idx]
            ss_res = float(np.sum((o_h - p_h) ** 2))
            ss_tot = float(np.sum((o_h - o_h.mean()) ** 2))
            nse_h = 1.0 - ss_res / ss_tot if ss_tot >= 1e-8 else float("nan")
            rmse_h = float(np.sqrt(np.mean((o_h - p_h) ** 2)))
            mae_h = float(np.mean(np.abs(o_h - p_h)))
            rse_h = float(ss_res / max(ss_tot, 1e-8))  # Relative Squared Error (1 - NSE)
            results[label] = {
                "nse": round(nse_h, 4),
                "rmse": round(rmse_h, 2),
                "mae": round(mae_h, 2),
                "rse": round(rse_h, 4),
            }
    return results




def picp(obs: np.ndarray, pred_low: np.ndarray, pred_high: np.ndarray) -> dict:
    """PICP (Prediction Interval Coverage Probability) -- ty le % thoi diem
    gia tri thuc te nam trong [pred_low, pred_high] (vd P5-P95). Neu dung
    dung P5/P95, ly tuong PICP ~= 0.90 -- KHONG phai 1.0, vi ~10% thuc te
    vuot ngoai dai la BINH THUONG neu model hieu chinh (calibrate) dung,
    khong phai loi. Tra ve them do rong trung binh cua dai (sharpness):
    PICP cao nhung dai qua rong thi khong co y nghia thuc te (model chi
    "cheat" bang cach du bao khoang rat lon) -- can nhin ca 2 chi so cung luc.
    """
    obs = np.asarray(obs)
    pred_low = np.asarray(pred_low)
    pred_high = np.asarray(pred_high)
    inside = (obs >= pred_low) & (obs <= pred_high)
    width = pred_high - pred_low
    return {
        "picp": round(float(inside.mean()), 4),
        "mean_interval_width": round(float(width.mean()), 2),
    }


def detect_flood_events(
    obs: np.ndarray,          # (T,) chuỗi quan trắc liên tục (1 lead-time cố định)
    threshold: float,
    min_separation: int = 24,
) -> list:
    """Tách các trận lũ riêng lẻ khỏi 1 chuỗi quan trắc liên tục."""
    T = len(obs)
    candidate = np.where(obs >= threshold)[0]
    if len(candidate) == 0:
        return []

    peak_indices = []
    i = 0
    while i < len(candidate):
        j = i
        while j + 1 < len(candidate) and candidate[j + 1] - candidate[j] <= min_separation:
            j += 1
        segment = candidate[i:j + 1]
        peak_indices.append(int(segment[np.argmax(obs[segment])]))
        i = j + 1

    events = []
    for peak_idx in peak_indices:
        start = peak_idx
        while start > 0 and obs[start - 1] <= obs[start]:
            start -= 1
        end = peak_idx
        while end + 1 < T and obs[end + 1] <= obs[end]:
            end += 1
        events.append({"start": start, "peak": peak_idx, "end": end})
    return events


def flood_event_diagnostics(
    obs: np.ndarray,
    pred: np.ndarray,
    threshold: float,
    min_separation: int = 24,
    peak_re_tolerance: float = 0.2,
) -> dict:
    """Chẩn đoán từng trận lũ riêng lẻ (NSE + sai số đỉnh + QA pass rate)."""
    events = detect_flood_events(obs, threshold, min_separation)
    if not events:
        return {
            "n_events": 0, "mean_event_nse": float("nan"),
            "peak_re_mean": float("nan"), "qa_pass_rate": float("nan"),
            "events": [],
        }

    details = []
    for ev in events:
        s, p, e = ev["start"], ev["peak"], ev["end"]
        o_seg = obs[s:e + 1]
        p_seg = pred[s:e + 1]
        ev_nse = nse_single(o_seg, p_seg)

        obs_peak = float(obs[p])
        pred_peak_in_window = float(p_seg.max()) if len(p_seg) else float("nan")
        re = abs(pred_peak_in_window - obs_peak) / max(obs_peak, 1e-6)

        details.append({
            "start": s, "peak": p, "end": e,
            "obs_peak": round(obs_peak, 2),
            "pred_peak": round(pred_peak_in_window, 2),
            "peak_re": round(re, 4),
            "event_nse": round(ev_nse, 4) if not np.isnan(ev_nse) else None,
            "qa_pass": bool(re < peak_re_tolerance),
        })

    valid_nse = [d["event_nse"] for d in details if d["event_nse"] is not None]
    return {
        "n_events": len(details),
        "mean_event_nse": round(float(np.mean(valid_nse)), 4) if valid_nse else float("nan"),
        "peak_re_mean": round(float(np.mean([d["peak_re"] for d in details])), 4),
        "qa_pass_rate": round(float(np.mean([d["qa_pass"] for d in details])), 4),
        "events": details,
    }


## Dataset Loader

In [ ]:
# data/reservoir_dataset.py
"""PyTorch Dataset cho 1 hồ — load v2_*.npy do data/dataset_builder.py sinh ra."""

import os
import numpy as np
import torch
from torch.utils.data import Dataset


class ReservoirDataset(Dataset):
    """
    Load datasets/<reservoir_key>/v2_*.npy.

    Mỗi item: (x_hindcast, x_nwp, y, station_rain, station_mask)
      x_hindcast   : (hindcast_len, n_hindcast_features) float32
      x_nwp        : (forecast_len, n_nwp_features)       float32
      y            : (forecast_len,)                       float32 — sqrt(inflow)
      station_rain : (hindcast_len, max_stations)          float32 — 0 nếu chưa build
      station_mask : (hindcast_len, max_stations)          bool    — False nếu chưa build

    station_rain/station_mask CHỈ có ý nghĩa khi config.use_station_attention=True
    (xem models/flood_lstm_v2.py). Luôn trả về đủ 5 phần tử để vòng lặp train/val/
    test không cần if/else riêng.
    """

    def __init__(self, data_dir: str, inflow_cap_sqrt: float | None = None, max_stations: int = 7):
        self.X_hind = np.load(os.path.join(data_dir, "v2_X_hindcast.npy"), mmap_mode="r")
        self.X_nwp  = np.load(os.path.join(data_dir, "v2_X_nwp.npy"),      mmap_mode="r")
        self.y      = np.load(os.path.join(data_dir, "v2_y.npy"),           mmap_mode="r")

        ts_path = os.path.join(data_dir, "v2_timestamps.npy")
        self.timestamps = np.load(ts_path) if os.path.exists(ts_path) else None

        rain_path   = os.path.join(data_dir, "v2_station_rain.npy")
        mask_path   = os.path.join(data_dir, "v2_station_mask.npy")
        prior_path  = os.path.join(data_dir, "v2_station_prior_weights.npy")
        self.has_station_data = os.path.exists(rain_path) and os.path.exists(mask_path)
        if self.has_station_data:
            self.station_rain = np.load(rain_path, mmap_mode="r")
            self.station_mask = np.load(mask_path, mmap_mode="r")
            self.max_stations = self.station_rain.shape[-1]
            self.station_prior_weights = (
                np.load(prior_path).tolist() if os.path.exists(prior_path) else None
            )
        else:
            self.station_rain = None
            self.station_mask = None
            self.max_stations = max_stations
            self.station_prior_weights = None

        self.inflow_cap_sqrt = inflow_cap_sqrt

        print(
            f"ReservoirDataset({data_dir}): {len(self):,} samples | "
            f"hindcast={self.X_hind.shape[1]}h | forecast={self.y.shape[1]}h | "
            f"station_attention_data={'ON' if self.has_station_data else 'OFF'}"
        )

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        y = np.array(self.y[idx], copy=True, dtype=np.float32)
        if self.inflow_cap_sqrt is not None:
            y = np.clip(y, 0.0, self.inflow_cap_sqrt)

        x_hind = torch.from_numpy(np.array(self.X_hind[idx], copy=False)).float()
        x_nwp  = torch.from_numpy(np.array(self.X_nwp[idx],  copy=False)).float()
        y_t    = torch.from_numpy(y).float()

        if self.has_station_data:
            station_rain = torch.from_numpy(np.array(self.station_rain[idx], copy=False)).float()
            station_mask = torch.from_numpy(np.array(self.station_mask[idx], copy=False)).bool()
        else:
            T_h = x_hind.shape[0]
            station_rain = torch.zeros(T_h, self.max_stations, dtype=torch.float32)
            station_mask = torch.zeros(T_h, self.max_stations, dtype=torch.bool)

        return x_hind, x_nwp, y_t, station_rain, station_mask


## Single Reservoir Training Function

In [ ]:
# training/train_reservoir.py
"""
Training script cho 1 hồ — adapt từ lstm_service/training/train_v2.py, bỏ
reservoir_idx/embedding, chỉ 1 nguồn NWP.

Chạy: python main_train.py --rid <id>   (từ LSTM_Py_Backend_v2/)
Hoặc: python -m training.train_reservoir --rid <id>

Giữ nguyên các cải tiến đã làm ở phiên trước (áp dụng cho lstm_service):
  - Seed cố định (torch/numpy/random + cudnn.deterministic)
  - Flood event oversampling (top 5%/1%)
  - NSE theo lead-time (nse_per_horizon) + chẩn đoán từng trận lũ
    (flood_event_diagnostics) ở bước đánh giá test cuối
  - Teacher forcing annealing, AMP fp16, cosine LR + warmup
"""

import os
import math
import json
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm









SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def nse_np(obs: np.ndarray, pred: np.ndarray) -> float:
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    return float("nan") if ss_tot < 1e-8 else 1.0 - ss_res / ss_tot


def compute_metrics(preds: torch.Tensor, targets: torch.Tensor, med_idx: int) -> dict:
    """preds: (N,T,Q) sqrt space, targets: (N,T) sqrt space -> metrics trên m3/s."""
    pred_med = preds[:, :, med_idx]
    pred_raw   = (pred_med ** 2).numpy()
    target_raw = (targets ** 2).numpy()

    mae  = float(np.mean(np.abs(pred_raw - target_raw)))
    rmse = float(np.sqrt(np.mean((pred_raw - target_raw) ** 2)))
    nse  = nse_np(target_raw.flatten(), pred_raw.flatten())
    r2   = nse  # R2 coefficient of determination is equal to NSE in hydrology evaluation
    return {"mae": mae, "rmse": rmse, "nse": nse, "r2": r2}



def train_reservoir(rid: int, cfg: ReservoirLSTMConfig = None, data_dir: str = None,
                     init_checkpoint: str = None):
    if rid not in RESERVOIRS:
        raise ValueError(f"rid={rid} không có trong config/reservoirs.py")
    info = RESERVOIRS[rid]
    reservoir_key = info["name"].replace(" ", "_")

    set_seed(SEED)

    cfg = cfg or ReservoirLSTMConfig(rid=rid, reservoir_name=info["name"])
    cfg.artifacts_dir = os.path.join(cfg.artifacts_dir, reservoir_key)
    os.makedirs(cfg.artifacts_dir, exist_ok=True)

    data_dir = data_dir or os.path.join("datasets", reservoir_key)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[{rid}] {info['name']}  |  Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    # ── Dataset ────────────────────────────────────────────────────────────────
    cap_m3s = INFLOW_CAPS_M3S.get(rid)
    cap_sqrt = math.sqrt(cap_m3s) if cap_m3s else None
    dataset = ReservoirDataset(data_dir, inflow_cap_sqrt=cap_sqrt, max_stations=cfg.max_stations)
    if cfg.use_station_attention and not dataset.has_station_data:
        print(
            "  WARNING: use_station_attention=True nhưng không tìm thấy "
            "v2_station_rain.npy/v2_station_mask.npy — model sẽ chạy với "
            "station_rain=0 (không học được gì thêm)."
        )

    if dataset.timestamps is None:
        raise FileNotFoundError(f"{data_dir}/v2_timestamps.npy không tìm thấy.")

    ts = dataset.timestamps
    train_end  = np.datetime64(cfg.train_end, "s")
    val_start  = np.datetime64(cfg.val_start, "s")
    val_end    = np.datetime64(cfg.val_end, "s")
    test_start = np.datetime64(cfg.test_start, "s")

    all_idx = np.arange(len(ts))
    train_idx = all_idx[ts < val_start].tolist()
    val_idx   = all_idx[(ts >= val_start) & (ts < val_end)].tolist()
    test_idx  = all_idx[ts >= test_start].tolist()

    print(f"Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}")
    if len(train_idx) < 100 or len(val_idx) < 10 or len(test_idx) < 10:
        print("  WARNING: Rất ít sample ở 1 trong các tập — kiểm tra lại coverage dữ liệu của hồ này.")

    # Flood oversampling (theo % gia tri cuc tri -- chi bat diem dinh rieng le)
    train_y_max = dataset.y[train_idx].max(axis=1)
    thr95 = float(np.percentile(train_y_max, 95))
    thr99 = float(np.percentile(train_y_max, 99))
    idx95 = [train_idx[i] for i in np.where(train_y_max >= thr95)[0]]
    idx99 = [train_idx[i] for i in np.where(train_y_max >= thr99)[0]]

    # Oversampling theo MUA MUA (thang 8-12): bo sung cho oversample theo %
    # cuc tri o tren -- day nhan ban CA giai doan mua mua (khong chi diem dinh
    # rieng le) de model hoc ky hon dang tang/giam cua tran lu.
    train_months = dataset.timestamps[train_idx].astype("datetime64[M]").astype(int) % 12 + 1
    idx_rainy = [train_idx[i] for i in np.where(np.isin(train_months, cfg.rainy_season_months))[0]]

    oversampled = (
        train_idx + idx95 * cfg.oversample_p95_factor + idx99 * cfg.oversample_p99_factor
        + idx_rainy * cfg.oversample_rainy_season_factor
    )
    print(f"Oversampling: top5%={len(idx95):,}x{cfg.oversample_p95_factor} | "
          f"top1%={len(idx99):,}x{cfg.oversample_p99_factor} | "
          f"mua_mua(T8-12)={len(idx_rainy):,}x{cfg.oversample_rainy_season_factor} | "
          f"total={len(oversampled):,}")

    n_w = 2 if device.type == "cuda" else 0
    _kw = dict(num_workers=n_w, pin_memory=(device.type == "cuda"), persistent_workers=(n_w > 0))
    train_loader = DataLoader(Subset(dataset, oversampled), batch_size=cfg.batch_size, shuffle=True, **_kw)
    val_loader   = DataLoader(Subset(dataset, val_idx),    batch_size=cfg.batch_size, shuffle=False, **_kw)
    test_loader  = DataLoader(Subset(dataset, test_idx),   batch_size=cfg.batch_size, shuffle=False, num_workers=n_w)

    # ── Model ──────────────────────────────────────────────────────────────────
    model = ReservoirLSTM(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"ReservoirLSTM parameters: {n_params:,}")

    if init_checkpoint:
        state = torch.load(init_checkpoint, map_location=device)
        # Checkpoint tu Phase 1 (pretrain pooled) co the KHAC kien truc voi model
        # hien tai theo 2 kieu: (a) key hoan toan moi (station_attn.* -- Phase 1
        # khong bao gio bat use_station_attention), (b) key TON TAI o ca 2 ben
        # nhung LECH SHAPE (hindcast_proj.0.* -- input dim +1 vi noi them
        # station_extra). strict=False CHI xu ly truong hop (a), van raise loi
        # cho truong hop (b) -- phai loc thu cong truoc khi load.
        own_state = model.state_dict()
        compatible, skipped_shape = {}, []
        for k, v in state.items():
            if k in own_state and own_state[k].shape == v.shape:
                compatible[k] = v
            elif k in own_state:
                skipped_shape.append((k, tuple(v.shape), tuple(own_state[k].shape)))
        missing, unexpected = model.load_state_dict(compatible, strict=False)
        print(f"  Warm-start: loaded pretrained weights from {init_checkpoint}")
        if skipped_shape:
            print(f"    (bo qua -- lech shape, khoi tao moi): {skipped_shape}")
        if missing:
            print(f"    (khoi tao moi -- khong co trong checkpoint): {missing}")
        if unexpected:
            print(f"    (bo qua -- co trong checkpoint nhung model khong dung): {unexpected}")

    if cfg.use_station_attention and getattr(dataset, "station_prior_weights", None):
        model.station_attn.init_prior(dataset.station_prior_weights)
        print(f"  station_attn.init_prior({[round(w, 3) for w in dataset.station_prior_weights]})")

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    def lr_lambda(epoch):
        if epoch < cfg.warmup_epochs:
            return float(epoch + 1) / cfg.warmup_epochs
        progress = (epoch - cfg.warmup_epochs) / max(cfg.epochs - cfg.warmup_epochs, 1)
        return max(0.05, 0.5 * (1 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    use_amp = device.type == "cuda"
    amp_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print(f"AMP: {'ON' if use_amp else 'OFF'}")

    best_val = float("inf")
    patience_cnt = 0
    history = []
    ckpt_path = os.path.join(cfg.artifacts_dir, "reservoir_lstm.pt")

    for epoch in range(cfg.epochs):
        tf_ratio = cfg.teacher_forcing_ratio(epoch)

        model.train()
        train_loss = 0.0
        for x_hind, x_nwp, y_b, station_rain, station_mask in tqdm(
            train_loader, desc=f"[{reservoir_key}] Epoch {epoch+1}/{cfg.epochs}", leave=False
        ):
            x_hind = x_hind.to(device)
            x_nwp  = x_nwp.to(device)
            y_b    = y_b.to(device)
            station_rain = station_rain.to(device)
            station_mask = station_mask.to(device)

            if cfg.target_noise_std > 0:
                y_noisy = y_b + torch.randn_like(y_b) * cfg.target_noise_std
            else:
                y_noisy = y_b

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp):
                preds = model(
                    x_hind, x_nwp, teacher_forcing_ratio=tf_ratio, y_true_sqrt=y_noisy,
                    station_rain=station_rain, station_mask=station_mask,
                )
                loss = quantile_loss_v2(preds, y_noisy, cfg.quantiles,
                                         horizon_decay=cfg.horizon_decay,
                                         coverage_weight=cfg.coverage_weight,
                                         peak_weight=cfg.peak_weight)

            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            amp_scaler.step(optimizer)
            amp_scaler.update()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # ── Validation ─────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        all_preds, all_targets = [], []
        with torch.no_grad():
            for x_hind, x_nwp, y_b, station_rain, station_mask in val_loader:
                x_hind = x_hind.to(device)
                x_nwp  = x_nwp.to(device)
                y_b    = y_b.to(device)
                station_rain = station_rain.to(device)
                station_mask = station_mask.to(device)

                with torch.amp.autocast("cuda", enabled=use_amp):
                    preds = model(
                        x_hind, x_nwp, teacher_forcing_ratio=0.0,
                        station_rain=station_rain, station_mask=station_mask,
                    )
                    val_loss += quantile_loss_v2(preds, y_b, cfg.quantiles,
                                                  horizon_decay=cfg.horizon_decay,
                                                  coverage_weight=cfg.coverage_weight,
                                                  peak_weight=cfg.peak_weight).item()

                all_preds.append(preds.cpu())
                all_targets.append(y_b.cpu())

        val_loss /= len(val_loader)
        preds_cat   = torch.cat(all_preds)
        targets_cat = torch.cat(all_targets)
        m = compute_metrics(preds_cat, targets_cat, cfg.median_idx)
        scheduler.step()
        lr_now = optimizer.param_groups[0]["lr"]

        print(
            f"[{reservoir_key}] Epoch {epoch+1:3d} | LR {lr_now:.2e} | TF {tf_ratio:.2f} | "
            f"Train {train_loss:.4f} | Val {val_loss:.4f} | "
            f"NSE {m['nse']:.3f} | MAE {m['mae']:.1f} | RMSE {m['rmse']:.1f} m3/s"
        )

        history.append({"epoch": epoch + 1, "lr": lr_now, "train_loss": train_loss,
                         "val_loss": val_loss, **m})

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), ckpt_path)
            torch.save(cfg, os.path.join(cfg.artifacts_dir, "config.pt"))
            patience_cnt = 0
            print("  Best model saved")
        else:
            patience_cnt += 1

        if patience_cnt >= cfg.patience:
            print("Early stopping.")
            break

    # ── Test Evaluation ─────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print(f"ĐÁNH GIÁ TẬP TEST (HOLDOUT) — {info['name']}")
    print("=" * 70)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_hind, x_nwp, y_b, station_rain, station_mask in test_loader:
            preds = model(
                x_hind.to(device), x_nwp.to(device), teacher_forcing_ratio=0.0,
                station_rain=station_rain.to(device), station_mask=station_mask.to(device),
            )
            all_preds.append(preds.cpu())
            all_targets.append(y_b)

    preds_cat   = torch.cat(all_preds)
    targets_cat = torch.cat(all_targets)
    m = compute_metrics(preds_cat, targets_cat, cfg.median_idx)

    # NSE theo lead-time + chẩn đoán từng trận lũ
    preds_np   = (preds_cat[:, :, cfg.median_idx] ** 2).numpy()
    targets_np = (targets_cat ** 2).numpy()
    np.save(os.path.join(cfg.artifacts_dir, "test_preds.npy"), preds_np)
    np.save(os.path.join(cfg.artifacts_dir, "test_targets.npy"), targets_np)
    m["kge"] = round(kge_single(targets_np.reshape(-1), preds_np.reshape(-1)), 4)

    pred_low_np  = (preds_cat[:, :, 0]  ** 2).numpy()   # P5
    pred_high_np = (preds_cat[:, :, -1] ** 2).numpy()   # P95
    picp_result = picp(targets_np.reshape(-1), pred_low_np.reshape(-1), pred_high_np.reshape(-1))
    m["picp_p5_p95"] = picp_result["picp"]
    m["mean_interval_width"] = picp_result["mean_interval_width"]

    print(f"NSE={m['nse']:.4f}  KGE={m['kge']:.4f}  MAE={m['mae']:.2f} m3/s  RMSE={m['rmse']:.2f} m3/s")
    print(f"PICP(P5-P95)={m['picp_p5_p95']:.4f} (ly tuong ~0.90)  "
          f"Do rong dai trung binh={m['mean_interval_width']:.1f} m3/s")

    horizon_metrics = metrics_at_specific_horizons(preds_np, targets_np, horizons=[3, 6, 12, 24])
    m["horizons"] = horizon_metrics

    horizon_rows = nse_per_horizon(preds_np, targets_np, group_hours=6)
    for h in horizon_rows:
        print(f"  lead {h['hour_range']:>8}  NSE={h['nse']}")
    for h_str, h_vals in horizon_metrics.items():
        print(f"  [Mốc {h_str:>3}] NSE={h_vals['nse']:.4f}  RMSE={h_vals['rmse']:.1f} m3/s  RSE={h_vals['rse']:.4f}")


    obs_series, pred_series = extract_lead_time_series(preds_np, targets_np, lead_idx=cfg.forecast_len - 1)
    event_diag = {"n_events": 0}
    if len(obs_series) > 10 and obs_series.max() > 0:
        thr = float(np.percentile(obs_series, 90))
        event_diag = flood_event_diagnostics(obs_series, pred_series, threshold=thr)
        print(f"  [lead={cfg.forecast_len}h] n_events={event_diag['n_events']}  "
              f"NSE_event={event_diag.get('mean_event_nse')}  "
              f"peak_RE={event_diag.get('peak_re_mean')}  QA={event_diag.get('qa_pass_rate')}")

    # ── Save kết quả ───────────────────────────────────────────────────────────
    pd.DataFrame(history).to_excel(os.path.join(cfg.artifacts_dir, "lich_su_training.xlsx"), index=False)
    pd.DataFrame(horizon_rows).to_excel(os.path.join(cfg.artifacts_dir, "nse_theo_gio.xlsx"), index=False)

    with open(os.path.join(cfg.artifacts_dir, "metrics_test.json"), "w", encoding="utf-8") as f:
        json.dump({"reservoir": info["name"], "rid": rid, **m,
                    "event_diagnostics": event_diag}, f, ensure_ascii=False, indent=2)

    print(f"\nSaved: {cfg.artifacts_dir}/reservoir_lstm.pt | lich_su_training.xlsx | "
          f"nse_theo_gio.xlsx | metrics_test.json")
    return m


## Pooled & River Branch Training Functions

In [ ]:
# training/pretrain_pooled.py
"""
Pretrain 1 model "nền" trên dữ liệu GỘP cả 16 hồ (không phân biệt hồ nào),
dùng làm điểm khởi tạo (warm-start) cho fine-tune riêng từng hồ sau đó
(training/train_reservoir.py::train_reservoir(..., init_checkpoint=...)).

Vì sao cần bước này: mỗi hồ train riêng chỉ có ~35,000 giờ dữ liệu — rất ít so
với 1.1M tham số của ReservoirLSTM (NSE thực tế train-riêng thấp hơn nhiều so
với model chung 556K mẫu của lstm_service). Pretrain trên dữ liệu gộp (quy mô
giống model chung) để học pattern mưa->dòng chảy tổng quát trước, sau đó
fine-tune ngắn (LR thấp, ít epoch) riêng từng hồ để đặc hiệu hóa — vẫn giữ lợi
ích "không lẫn dữ liệu xấu giữa các hồ" ở bước fine-tune, không mất lợi ích
"nhiều dữ liệu" ở bước pretrain.

Gộp bằng torch.utils.data.ConcatDataset ("gộp ảo", KHÔNG ghi bản sao dữ liệu ra
đĩa) — mỗi ReservoirDataset con vẫn đọc trực tiếp (mmap) từ thư mục nguồn của
chính nó. Chỉ concatenate 2 mảng NHỎ trong RAM (timestamps, y — vài chục MB
cho cả 16 hồ) để tính train/val split + oversampling; X_hindcast/X_nwp (phần
nặng, ~24GB) không bao giờ bị copy/gộp. Quan trọng trên Kaggle vì
/kaggle/working thường bị giới hạn dung lượng (~20GB) — ghi hẳn bản gộp 25GB
ra đó sẽ tràn đĩa và làm kernel chết.

Chạy: python -m training.pretrain_pooled
Hoặc: from training.pretrain_pooled import pretrain_pooled; pretrain_pooled(data_dirs=[...])
"""
import os
import math
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset, ConcatDataset
from tqdm import tqdm










def _default_data_dirs(out_root: str = "datasets") -> list:
    dirs = []
    for info in RESERVOIRS.values():
        d = os.path.join(out_root, info["name"].replace(" ", "_"))
        if os.path.exists(os.path.join(d, "v2_X_hindcast.npy")):
            dirs.append(d)
    return dirs


def pretrain_pooled(
    data_dirs: list = None,
    cfg: ReservoirLSTMConfig = None,
    epochs: int = None,
    artifacts_dir: str = "artifacts/_POOLED_PRETRAIN",
) -> str:
    """
    Train 1 ReservoirLSTM trên dữ liệu gộp ẢO của nhiều hồ (ConcatDataset).
    data_dirs: list thư mục datasets/<reservoir_key>/ — mặc định: tự quét
    datasets/<Ten_Ho>/ của tất cả hồ đã build trong config/reservoirs.py.
    Trả về path checkpoint tốt nhất.
    """
    set_seed(SEED)
    cfg = cfg or ReservoirLSTMConfig(rid=0, reservoir_name="POOLED_PRETRAIN")
    if epochs:
        cfg.epochs = epochs
    os.makedirs(artifacts_dir, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[PRETRAIN POOLED]  Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    data_dirs = data_dirs or _default_data_dirs()
    if not data_dirs:
        raise RuntimeError("Không có thư mục dữ liệu hồ nào để gộp (data_dirs rỗng).")

    # inflow_cap_sqrt=None: mỗi hồ đã được cap/sqrt đúng theo INFLOW_CAPS_M3S
    # riêng của nó LÚC BUILD — không áp lại 1 cap chung cho dữ liệu gộp.
    per_reservoir = [ReservoirDataset(d, inflow_cap_sqrt=None, max_stations=cfg.max_stations)
                      for d in data_dirs]
    pooled = ConcatDataset(per_reservoir)
    print(f"Pooled từ {len(per_reservoir)} hồ -> {len(pooled):,} mẫu (ConcatDataset, không copy X_hind/X_nwp)")

    # ── Chỉ concatenate 2 mảng NHỎ (timestamps, y) trong RAM để tính split/oversample ──
    ts = np.concatenate([np.asarray(ds.timestamps) for ds in per_reservoir], axis=0)
    y_all = np.concatenate([np.asarray(ds.y) for ds in per_reservoir], axis=0)

    val_start = np.datetime64(cfg.val_start, "s")
    val_end   = np.datetime64(cfg.val_end, "s")

    all_idx = np.arange(len(ts))
    train_idx = all_idx[ts < val_start].tolist()
    val_idx   = all_idx[(ts >= val_start) & (ts < val_end)].tolist()
    print(f"Pooled Train: {len(train_idx):,} | Val: {len(val_idx):,}")

    train_y_max = y_all[train_idx].max(axis=1)
    thr95 = float(np.percentile(train_y_max, 95))
    thr99 = float(np.percentile(train_y_max, 99))
    idx95 = [train_idx[i] for i in np.where(train_y_max >= thr95)[0]]
    idx99 = [train_idx[i] for i in np.where(train_y_max >= thr99)[0]]

    # Oversampling theo MUA MUA (thang 8-12) -- xem ghi chu trong train_reservoir.py
    train_months = ts[train_idx].astype("datetime64[M]").astype(int) % 12 + 1
    idx_rainy = [train_idx[i] for i in np.where(np.isin(train_months, cfg.rainy_season_months))[0]]

    oversampled = (
        train_idx + idx95 * cfg.oversample_p95_factor + idx99 * cfg.oversample_p99_factor
        + idx_rainy * cfg.oversample_rainy_season_factor
    )
    print(f"Oversampling: top5%={len(idx95):,}x{cfg.oversample_p95_factor} | "
          f"top1%={len(idx99):,}x{cfg.oversample_p99_factor} | "
          f"mua_mua(T8-12)={len(idx_rainy):,}x{cfg.oversample_rainy_season_factor} | "
          f"total={len(oversampled):,}")

    n_w = 2 if device.type == "cuda" else 0
    _kw = dict(num_workers=n_w, pin_memory=(device.type == "cuda"), persistent_workers=(n_w > 0))
    train_loader = DataLoader(Subset(pooled, oversampled), batch_size=cfg.batch_size, shuffle=True, **_kw)
    val_loader   = DataLoader(Subset(pooled, val_idx),    batch_size=cfg.batch_size, shuffle=False, **_kw)

    model = ReservoirLSTM(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"ReservoirLSTM parameters: {n_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    def lr_lambda(epoch):
        if epoch < cfg.warmup_epochs:
            return float(epoch + 1) / cfg.warmup_epochs
        progress = (epoch - cfg.warmup_epochs) / max(cfg.epochs - cfg.warmup_epochs, 1)
        return max(0.05, 0.5 * (1 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    use_amp = device.type == "cuda"
    amp_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print(f"AMP: {'ON' if use_amp else 'OFF'}")

    best_val = float("inf")
    patience_cnt = 0
    ckpt_path = os.path.join(artifacts_dir, "pretrain_pooled.pt")

    for epoch in range(cfg.epochs):
        tf_ratio = cfg.teacher_forcing_ratio(epoch)

        model.train()
        train_loss = 0.0
        for x_hind, x_nwp, y_b, station_rain, station_mask in tqdm(
            train_loader, desc=f"[POOLED] Epoch {epoch+1}/{cfg.epochs}", leave=False
        ):
            x_hind, x_nwp, y_b = x_hind.to(device), x_nwp.to(device), y_b.to(device)
            station_rain, station_mask = station_rain.to(device), station_mask.to(device)
            y_noisy = y_b + torch.randn_like(y_b) * cfg.target_noise_std if cfg.target_noise_std > 0 else y_b

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp):
                preds = model(
                    x_hind, x_nwp, teacher_forcing_ratio=tf_ratio, y_true_sqrt=y_noisy,
                    station_rain=station_rain, station_mask=station_mask,
                )
                loss = quantile_loss_v2(preds, y_noisy, cfg.quantiles,
                                         horizon_decay=cfg.horizon_decay,
                                         coverage_weight=cfg.coverage_weight,
                                         peak_weight=cfg.peak_weight)

            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            amp_scaler.step(optimizer)
            amp_scaler.update()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        all_preds, all_targets = [], []
        with torch.no_grad():
            for x_hind, x_nwp, y_b, station_rain, station_mask in val_loader:
                x_hind, x_nwp, y_b = x_hind.to(device), x_nwp.to(device), y_b.to(device)
                station_rain, station_mask = station_rain.to(device), station_mask.to(device)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    preds = model(
                        x_hind, x_nwp, teacher_forcing_ratio=0.0,
                        station_rain=station_rain, station_mask=station_mask,
                    )
                    val_loss += quantile_loss_v2(preds, y_b, cfg.quantiles,
                                                  horizon_decay=cfg.horizon_decay,
                                                  coverage_weight=cfg.coverage_weight,
                                                  peak_weight=cfg.peak_weight).item()
                all_preds.append(preds.cpu())
                all_targets.append(y_b.cpu())

        val_loss /= len(val_loader)
        preds_cat, targets_cat = torch.cat(all_preds), torch.cat(all_targets)
        m = compute_metrics(preds_cat, targets_cat, cfg.median_idx)
        scheduler.step()
        lr_now = optimizer.param_groups[0]["lr"]

        print(
            f"[POOLED] Epoch {epoch+1:3d} | LR {lr_now:.2e} | TF {tf_ratio:.2f} | "
            f"Train {train_loss:.4f} | Val {val_loss:.4f} | "
            f"NSE {m['nse']:.3f} | MAE {m['mae']:.1f} | RMSE {m['rmse']:.1f} m3/s"
        )

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), ckpt_path)
            patience_cnt = 0
            print("  Best pooled model saved")
        else:
            patience_cnt += 1

        if patience_cnt >= cfg.patience:
            print("Early stopping (pooled pretrain).")
            break

    print(f"\nPretrain xong -> {ckpt_path}")
    return ckpt_path


def evaluate_model_on_reservoir(
    ckpt_path: str,
    data_dir: str,
    cfg: ReservoirLSTMConfig = None,
    save_preds_path: str = None,
) -> dict:
    """Đánh giá 1 model checkpoint bất kỳ (vd: model lưu vực) trên tập test của 1 hồ cụ thể."""
    set_seed(SEED)
    cfg = cfg or ReservoirLSTMConfig()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = ReservoirDataset(data_dir, max_stations=cfg.max_stations)
    if dataset.timestamps is None:
        raise FileNotFoundError(f"{data_dir}/v2_timestamps.npy không tìm thấy.")

    ts = dataset.timestamps
    test_start = np.datetime64(cfg.test_start, "s")
    all_idx = np.arange(len(ts))
    test_idx = all_idx[ts >= test_start].tolist()

    n_w = 2 if device.type == "cuda" else 0
    test_loader = DataLoader(Subset(dataset, test_idx), batch_size=cfg.batch_size, shuffle=False, num_workers=n_w)

    model = ReservoirLSTM(cfg).to(device)
    state = torch.load(ckpt_path, map_location=device)
    # Loc key lech shape truoc khi load (vd danh gia checkpoint cu -- khong co
    # station_attn, hindcast_proj input dim khac -- bang cfg moi co
    # use_station_attention=True, hoac nguoc lai) -- strict=False khong tu xu ly
    # duoc truong hop nay (chi xu ly key thieu/thua, khong xu ly key lech shape).
    own_state = model.state_dict()
    compatible = {k: v for k, v in state.items() if k in own_state and own_state[k].shape == v.shape}
    model.load_state_dict(compatible, strict=False)
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_hind, x_nwp, y_b, station_rain, station_mask in test_loader:
            preds = model(
                x_hind.to(device), x_nwp.to(device), teacher_forcing_ratio=0.0,
                station_rain=station_rain.to(device), station_mask=station_mask.to(device),
            )
            all_preds.append(preds.cpu())
            all_targets.append(y_b)

    preds_cat = torch.cat(all_preds)
    targets_cat = torch.cat(all_targets)
    metrics = compute_metrics(preds_cat, targets_cat, cfg.median_idx)

    preds_np = (preds_cat[:, :, cfg.median_idx] ** 2).numpy()
    targets_np = (targets_cat ** 2).numpy()
    metrics["horizons"] = metrics_at_specific_horizons(preds_np, targets_np, horizons=[3, 6, 12, 24])
    metrics["kge"] = round(kge_single(targets_np.reshape(-1), preds_np.reshape(-1)), 4)

    pred_low_np  = (preds_cat[:, :, 0]  ** 2).numpy()   # P5
    pred_high_np = (preds_cat[:, :, -1] ** 2).numpy()   # P95
    picp_result = picp(targets_np.reshape(-1), pred_low_np.reshape(-1), pred_high_np.reshape(-1))
    metrics["picp_p5_p95"] = picp_result["picp"]
    metrics["mean_interval_width"] = picp_result["mean_interval_width"]

    if save_preds_path:
        os.makedirs(save_preds_path, exist_ok=True)
        np.save(os.path.join(save_preds_path, "basin_preds.npy"), preds_np)
        np.save(os.path.join(save_preds_path, "basin_targets.npy"), targets_np)
    return metrics



def train_river_branch_model(
    basin_name: str,
    rids: list = None,
    epochs: int = None,
    data_dirs_map: dict = None,
    artifacts_dir: str = None,
) -> str:
    """Train 1 model cho nguyên 1 nhánh sông (Vu Gia hoặc Thu Bồn)."""
    if rids is None:
        from config.reservoirs import RIVER_BASINS
        if basin_name not in RIVER_BASINS:
            raise ValueError(f"basin_name={basin_name} không hợp lệ. Chọn 'Vu Gia' hoặc 'Thu Bồn'.")
        rids = RIVER_BASINS[basin_name]

    if data_dirs_map is None:
        from config.reservoirs import RESERVOIRS
        data_dirs_map = {}
        for rid in rids:
            info = RESERVOIRS[rid]
            key = info["name"].replace(" ", "_")
            data_dirs_map[rid] = os.path.join("datasets", key)

    dirs = [data_dirs_map[rid] for rid in rids if rid in data_dirs_map and os.path.exists(data_dirs_map[rid])]
    if not dirs:
        raise RuntimeError(f"Không tìm thấy thư mục dữ liệu cho lưu vực {basin_name}")

    artifacts_dir = artifacts_dir or os.path.join("artifacts", f"_BRANCH_{basin_name.upper().replace(' ', '_')}")
    cfg = ReservoirLSTMConfig(rid=0, reservoir_name=f"NHANH_SONG_{basin_name.upper()}")

    print(f"\n========================================================")
    print(f"TRAIN MODEL NHÁNH SÔNG: {basin_name.upper()} ({len(dirs)} hồ)")
    print(f"========================================================")

    return pretrain_pooled(data_dirs=dirs, cfg=cfg, epochs=epochs, artifacts_dir=artifacts_dir)


## Pha 1 — Train Model cho 2 Nhánh Sông (Vu Gia & Thu Bồn)
Pooled Data từ 11 hồ Vu Gia và 5 hồ Thu Bồn, train trong 20 epochs với batch_size=256.


In [ ]:
BRANCH_CHECKPOINTS = {}
FORCE_RETRAIN = True

for basin_name, rids in RIVER_BASINS.items():
    basin_key = basin_name.upper().replace(" ", "_")
    artifacts_dir = f"{OUTPUT_ROOT}/_BRANCH_{basin_key}"
    ckpt_path = f"{artifacts_dir}/pretrain_pooled.pt"

    # TỰ ĐỘNG PHÁT HIỆN RESUME: Nếu model nhánh sông đã train trước đó -> SKIP train lại
    if not FORCE_RETRAIN and os.path.exists(ckpt_path):
        print(f"[SKIP TRAIN] Model nhánh sông '{basin_name}' đã tồn tại: {ckpt_path}. Bỏ qua train lại!")
        BRANCH_CHECKPOINTS[basin_name] = ckpt_path
        continue

    print("\n" + "=" * 70)
    print(f"TRAIN MODEL CHO NHÁNH SÔNG: {basin_name.upper()} ({len(rids)} HỒ) - 20 EPOCHS")
    print("=" * 70)

    data_dirs = []
    for rid in rids:
        key = RESERVOIRS[rid]["name"].replace(" ", "_")
        if key in RESERVOIR_DATA_DIRS:
            data_dirs.append(RESERVOIR_DATA_DIRS[key])

    if not data_dirs:
        print(f"Không tìm thấy thư mục data cho nhánh {basin_name}")
        continue

    cfg = ReservoirLSTMConfig(rid=0, reservoir_name=f"BRANCH_{basin_key}")
    cfg.epochs = 20
    cfg.patience = 6
    cfg.warmup_epochs = 2
    cfg.batch_size = 256

    # Train model lưu vực
    ckpt = pretrain_pooled(
        data_dirs=data_dirs,
        cfg=cfg,
        artifacts_dir=artifacts_dir,
    )
    BRANCH_CHECKPOINTS[basin_name] = ckpt
    print(f"Checkpoint {basin_name}: {ckpt}")


## Pha 2 — Đánh giá Model Nhánh Sông trên từng hồ
Dùng model nhánh sông (Vu Gia / Thu Bồn) đã train ở Pha 1 để đánh giá độ chính xác trên tập test của từng hồ thuộc nhánh đó.


In [ ]:
basin_eval_results = {}

for rid, info in RESERVOIRS.items():
    basin = info["river_basin"]
    ckpt = BRANCH_CHECKPOINTS.get(basin)
    key = info["name"].replace(" ", "_")
    data_dir = RESERVOIR_DATA_DIRS.get(key)

    if not ckpt or not data_dir:
        print(f"[SKIP] Bỏ qua {info['name']} vì thiếu model/data lưu vực")
        continue

    cfg = ReservoirLSTMConfig(rid=rid, reservoir_name=info["name"])
    cfg.batch_size = 256
    m_basin = evaluate_model_on_reservoir(
        ckpt, data_dir, cfg=cfg, save_preds_path=f"{OUTPUT_ROOT}/{key}"
    )
    basin_eval_results[rid] = m_basin

    # Lưu file json đánh giá mô hình nhánh sông ra ổ đĩa
    os.makedirs(f"{OUTPUT_ROOT}/{key}", exist_ok=True)
    with open(f"{OUTPUT_ROOT}/{key}_basin_eval.json", "w", encoding="utf-8") as f:
        json.dump(m_basin, f, ensure_ascii=False, indent=2)
    with open(f"{OUTPUT_ROOT}/{key}/metrics_basin.json", "w", encoding="utf-8") as f:
        json.dump(m_basin, f, ensure_ascii=False, indent=2)

    print(f"[{basin}] {info['name']} | NSE_basin: {m_basin['nse']:.4f} | RMSE: {m_basin['rmse']:.1f} m3/s | MAE: {m_basin['mae']:.1f} m3/s")


## Pha 3 — Fine-tune siêu tốc riêng từng hồ (Single Reservoir Training)
Warm-start từ model nhánh sông đã train ở Pha 1, fine-tune trong 20 epochs với batch_size=256 (~12-15 giây/hồ).


In [ ]:
single_eval_results = {}

for rid, info in RESERVOIRS.items():
    key = info["name"].replace(" ", "_")
    data_dir = RESERVOIR_DATA_DIRS.get(key)
    if not data_dir:
        continue

    basin = info["river_basin"]
    branch_ckpt = BRANCH_CHECKPOINTS.get(basin)
    cfg = ReservoirLSTMConfig(rid=rid, reservoir_name=info["name"])
    cfg.artifacts_dir = f"{OUTPUT_ROOT}/{key}"
    cfg.batch_size = 256
    # Kich ban 2: bat CHI o Phase 3 (fine-tune 1 ho co dinh) -- station slot
    # co y nghia nhat quan trong toan bo qua trinh train ho nay. KHONG bat o
    # Phase 1/2 (pool nhieu ho, moi ho co bo tram khac nhau -- xem ghi chu
    # models/station_attention.py). Can dataset da build lai voi
    # v2_station_rain.npy/v2_station_mask.npy (xem data/dataset_builder.py).
    cfg.use_station_attention = True

    # train_reservoir() tu noi them 1 cap thu muc con (os.path.join(cfg.artifacts_dir,
    # reservoir_key)) truoc khi luu -- phai kiem tra dung duong dan LONG NAY thi
    # logic resume/skip moi thuc su hoat dong (neu khong se luon retrain lai tu dau).
    metrics_file = f"{cfg.artifacts_dir}/{key}/metrics_test.json"
    ckpt_single = f"{cfg.artifacts_dir}/{key}/reservoir_lstm.pt"

    # TỰ ĐỘNG PHÁT HIỆN HỒ ĐÃ TRAIN XONG: Nếu có kết quả rồi -> Bỏ qua train lại, load luôn kết quả!
    if not FORCE_RETRAIN and os.path.exists(metrics_file):
        print(f"[SKIP / RESUME] Hồ {info['name']} đã train hoàn tất trước đó. Bỏ qua train lại!")
        with open(metrics_file, "r", encoding="utf-8") as f:
            single_eval_results[rid] = json.load(f)
        continue
    elif not FORCE_RETRAIN and os.path.exists(ckpt_single):
        print(f"[SKIP / RESUME] Tìm thấy checkpoint {ckpt_single}, load lại và đánh giá...")
        single_eval_results[rid] = evaluate_model_on_reservoir(ckpt_single, data_dir, cfg=cfg)
        continue

    print("\n" + "#" * 70)
    print(f"# FINE-TUNE HỒ ĐỘC LẬP: [{rid}] {info['name']} (Lưu vực {basin}) - toi da 30 EPOCHS")
    print("#" * 70)

    # Fine-tune voi LR THAP (1e-4, thay vi 3e-4 dung o pha pretrain) de tranh
    # "quen" trong so pretrain nhanh song; epochs/patience noi ra (30/8, thay
    # vi 10/4) de mo hinh co du thoi gian hoi tu sau warm-start. Cham hon
    # ban cu nhung early stopping (patience) van tu dung neu hoi tu som.
    if branch_ckpt is not None:
        cfg.lr = 1e-4
        cfg.epochs = 30
        cfg.patience = 8
        cfg.warmup_epochs = 3
        cfg.batch_size = 256

    m_single = train_reservoir(rid, cfg=cfg, data_dir=data_dir, init_checkpoint=branch_ckpt)
    single_eval_results[rid] = m_single



## Pha 4 — Tổng hợp & Tạo Bảng So Sánh Theo Dạng Yêu Cầu
Tổng hợp chỉ số từ cả 2 cách train và trình bày bảng kết quả theo đúng cấu trúc ảnh.


In [ ]:
table_rows = []

# Danh sách thứ tự hiển thị ưu tiên theo ảnh yêu cầu
custom_order = [
    "HO ZA HUNG", "HO DAK MI 3", "HO SONG BUNG 4", "HO DAK MI 2", "HO DAK MI 4",
    "HO SONG TRANH 4", "HO A VUONG", "HO SONG TRANH 3", "HO SONG TRANH 2",
    "HO SONG BUNG 2", "HO SONG CON 2", "HO KHE DIEN", "HO SONG BUNG 5",
    "HO SONG BUNG 6", "HO SONG BUNG 4A", "HO DAK MI 4C"
]

# Map name -> rid
name_to_rid = {info["name"]: rid for rid, info in RESERVOIRS.items()}

for res_name in custom_order:
    rid = name_to_rid.get(res_name)
    if not rid:
        continue

    info = RESERVOIRS[rid]
    basin = info["river_basin"]

    m_single = single_eval_results.get(rid, {})
    m_basin  = basin_eval_results.get(rid, {})

    nse_single_val = m_single.get("nse", float("nan"))
    nse_basin_val  = m_basin.get("nse", float("nan"))
    kge_single_val = m_single.get("kge", float("nan"))
    kge_basin_val  = m_basin.get("kge", float("nan"))
    picp_single_val = m_single.get("picp_p5_p95", float("nan"))
    picp_basin_val  = m_basin.get("picp_p5_p95", float("nan"))



    rmse_single = m_single.get("rmse", float("nan"))
    rmse_basin  = m_basin.get("rmse", float("nan"))
    mae_single  = m_single.get("mae", float("nan"))
    mae_basin   = m_basin.get("mae", float("nan"))

    # Lấy chỉ số theo từng mốc thời gian 3h, 6h, 12h, 24h, 3d, 7d
    h_single = m_single.get("horizons", {})
    h_basin  = m_basin.get("horizons", {})

    # Đánh giá so sánh
    if np.isnan(nse_single_val) and np.isnan(nse_basin_val):
        verdict = "Chưa đủ dữ liệu để so sánh"
        improvement = "Kiểm tra lại dữ liệu đầu vào và các năm quan trắc."
    elif np.isnan(nse_basin_val):
        verdict = f"Train riêng hồ đạt NSE={nse_single_val:.3f}, RMSE={rmse_single:.1f} m3/s."
        improvement = "Hồ có đặc trưng dòng chảy riêng biệt; mô hình hội tụ tốt."
    elif nse_single_val > nse_basin_val + 0.03:
        verdict = f"Train riêng tốt hơn (NSE {nse_single_val:.3f} vs {nse_basin_val:.3f}). RMSE: {rmse_single:.1f} vs {rmse_basin:.1f} m3/s."
        improvement = "Hồ có đặc trưng dòng chảy riêng biệt; ưu tiên fine-tune sâu hơn trên dữ liệu hồ này."
    elif nse_basin_val > nse_single_val + 0.03:
        verdict = f"Train theo nhánh tốt hơn (NSE {nse_basin_val:.3f} vs {nse_single_val:.3f}). RMSE: {rmse_basin:.1f} vs {rmse_single:.1f} m3/s."
        improvement = "Dữ liệu riêng của hồ ít; học chuyển giao (transfer learning) từ nhánh sông giúp mô hình tổng quát hóa tốt hơn."
    else:
        verdict = f"Tương đương nhau (NSE từng hồ: {nse_single_val:.3f}, NSE nhánh: {nse_basin_val:.3f})."
        improvement = "Có thể kết hợp Ensemble (trung bình trọng số) giữa model từng hồ và model nhánh sông."

    row = {
        "Hồ": info["name"],
        "Lưu vực sông": basin,
        "NSE theo train từng hồ": round(nse_single_val, 4) if not np.isnan(nse_single_val) else "N/A",
        "NSE train theo lưu vực sông ( train dữ liệu cho toàn bộ theo nhánh sông Vu Gia-Thu Bồn )": round(nse_basin_val, 4) if not np.isnan(nse_basin_val) else "N/A",
        "KGE theo train từng hồ": round(kge_single_val, 4) if not np.isnan(kge_single_val) else "N/A",
        "KGE train theo lưu vực sông": round(kge_basin_val, 4) if not np.isnan(kge_basin_val) else "N/A",
        "PICP(P5-P95) theo train từng hồ": round(picp_single_val, 4) if not np.isnan(picp_single_val) else "N/A",
        "PICP(P5-P95) train theo lưu vực sông": round(picp_basin_val, 4) if not np.isnan(picp_basin_val) else "N/A",
        "Đánh giá kết quả ( train theo cách nào cho chỉ số tốt hơn), thêm các chỉ số cần thiết": verdict,
        "Nếu có cải tiến thì cải tiến những gì để đạt kết quả tốt hơn": improvement,
    }

    # Thêm chỉ số chi tiết từng mốc 3h, 6h, 12h, 24h, 3d, 7d
    for horizon_key in ["3h", "6h", "12h", "24h", "3d", "7d"]:
        hs = h_single.get(horizon_key, {})
        hb = h_basin.get(horizon_key, {})
        row[f"NSE_TừngHồ_{horizon_key}"] = hs.get("nse", "N/A")
        row[f"RSE_TừngHồ_{horizon_key}"] = hs.get("rse", "N/A")
        row[f"RMSE_TừngHồ_{horizon_key}"] = hs.get("rmse", "N/A")

        row[f"NSE_Nhánh_{horizon_key}"] = hb.get("nse", "N/A")
        row[f"RSE_Nhánh_{horizon_key}"] = hb.get("rse", "N/A")
        row[f"RMSE_Nhánh_{horizon_key}"] = hb.get("rmse", "N/A")

    table_rows.append(row)

result_df = pd.DataFrame(table_rows)


# In ra bảng Markdown thu gọn
main_cols = [
    "Hồ", "Lưu vực sông", "NSE theo train từng hồ",
    "NSE train theo lưu vực sông ( train dữ liệu cho toàn bộ theo nhánh sông Vu Gia-Thu Bồn )",
    "KGE theo train từng hồ", "KGE train theo lưu vực sông",
    "PICP(P5-P95) theo train từng hồ", "PICP(P5-P95) train theo lưu vực sông",
    "Đánh giá kết quả ( train theo cách nào cho chỉ số tốt hơn), thêm các chỉ số cần thiết",
    "Nếu có cải tiến thì cải tiến những gì để đạt kết quả tốt hơn"
]
print("=" * 80)
print("BẢNG TỔNG HỢP KẾT QUẢ DANH NGHĨA CHÍNH:")
print("=" * 80)
print(result_df[main_cols].to_string(index=False))

# In ra bảng chi tiết theo mốc dự báo 3h, 6h, 12h, 24h
horizon_cols = ["Hồ", "Lưu vực sông"] + [c for c in result_df.columns if "_" in c]
print("\n" + "=" * 80)
print("BẢNG CHI TIẾT THEO TỪNG MỐC DỰ BÁO (3H, 6H, 12H, 24H):")
print("=" * 80)
print(result_df[horizon_cols].to_string(index=False))

# Lưu file Excel với 2 sheet
excel_path = f"{OUTPUT_ROOT}/bang_so_sanh_nse_nhanh_song.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    result_df[main_cols].to_excel(writer, sheet_name="Tong_Hop", index=False)
    result_df[horizon_cols].to_excel(writer, sheet_name="Chi_Tiet_Moc_Thoi_Gian", index=False)
    result_df.to_excel(writer, sheet_name="Day_Du_Toan_Bo", index=False)

print(f"\nĐã xuất kết quả so sánh ra file Excel (chứa cả sheet Tổng hợp và Chi tiết mốc 3h-24h): {excel_path}")


## Ghi chu tinh trang Kich ban 2 & Kich ban 4 (chua noi vao pipeline)

**Kich ban 2 -- Station Rain Attention**: code da day du (`StationRainAttention`,
flag `use_station_attention` trong `config/settings.py`) va da wire vao
`ReservoirLSTM`. NHUNG file du lieu tung tram (`v2_station_rain.npy`,
`v2_station_mask.npy`) **khong ton tai tren dia cho bat ky ho nao**. Bat
flag ma khong rebuild dataset se chi nhan toan so 0 -- khong hoc duoc gi.
De dung that: (1) dat `use_station_attention = True` trong `config/settings.py`,
(2) chay lai buoc build dataset (`data/dataset_builder.py`) de sinh cac file
`.npy` tren tu `Data_Tung_Ho_Ma_Tran_Rong/*.xlsx`, (3) train lai tu dau (doi
input dim cua model).

**Kich ban 4 -- Q_outflow ho thuong nguon lam feature**: da them config nen
`UPSTREAM_RESERVOIRS` (xem cell "config/reservoirs.py" o dau notebook) theo
dung quan he ban mo ta (Song Bung 4A/5/6 <- Song Bung 2, Song Bung 4, Dak Mi 2).
Day moi chi la du lieu cau hinh -- **chua** co code nao trong
`data/dataset_builder.py` thuc su lay Q_outflow cua ho thuong nguon, can
chinh theo dung khung gio (time-align) voi hindcast window cua ho ha luu,
roi ghep them vao tensor dau vao. Day la thay doi kien truc du lieu can
rebuild dataset + doi input dim, nen chua tu dong lam trong notebook nay --
can xac nhan them truoc khi trien khai vi anh huong toi toan bo 3 ho.


## Pha 5 -- Ensemble co trong so (Kich ban 5, code that thay vi chi goi y)

Pha 4 truoc day chi IN RA mot cau goi y ("co the ket hop ensemble") khi NSE
cua 2 cach train gan nhau, khong co dong code nao thuc su tinh trung binh
co trong so. Pha nay lam that: voi moi ho ma NSE(tung ho) va NSE(nhanh song)
chenh nhau duoi 0.03 (dung dieu kien Pha 4 da neu), tai lai du doan tho da
luu (`test_preds.npy` cua model fine-tune rieng, `basin_preds.npy` cua model
nhanh song danh gia tren cung ho do), tron theo trong so
`0.6 * du_doan_tung_ho + 0.4 * du_doan_nhanh_song`, roi tinh lai NSE/KGE/RMSE/MAE
tren ket qua tron. Ket qua chi co y nghia neu ca 2 model da chay va luu
file `.npy` (tuc la da chay Pha 2 va Pha 3 that, khong phai resume/skip).


In [ ]:
ENSEMBLE_WEIGHT_SINGLE = 0.6
ENSEMBLE_WEIGHT_BASIN = 0.4
NSE_CLOSE_THRESHOLD = 0.03

ensemble_results = {}

for rid, info in RESERVOIRS.items():
    key = info["name"].replace(" ", "_")
    res_dir = f"{OUTPUT_ROOT}/{key}"
    p_single_path = f"{res_dir}/test_preds.npy"
    t_single_path = f"{res_dir}/test_targets.npy"
    p_basin_path = f"{res_dir}/basin_preds.npy"
    t_basin_path = f"{res_dir}/basin_targets.npy"

    if not (os.path.exists(p_single_path) and os.path.exists(p_basin_path)):
        continue

    nse_single_val = single_eval_results.get(rid, {}).get("nse", float("nan"))
    nse_basin_val = basin_eval_results.get(rid, {}).get("nse", float("nan"))
    if np.isnan(nse_single_val) or np.isnan(nse_basin_val):
        continue
    if abs(nse_single_val - nse_basin_val) >= NSE_CLOSE_THRESHOLD:
        # 1 trong 2 model ro rang tot hon han -> khong ensemble, tranh pha loang
        continue

    preds_single = np.load(p_single_path)
    targets_single = np.load(t_single_path)
    preds_basin = np.load(p_basin_path)
    targets_basin = np.load(t_basin_path)

    if preds_single.shape != preds_basin.shape:
        print(f"[SKIP ENSEMBLE] {info['name']}: shape khac nhau giua 2 tap test -- bo qua.")
        continue

    preds_ensemble = ENSEMBLE_WEIGHT_SINGLE * preds_single + ENSEMBLE_WEIGHT_BASIN * preds_basin

    nse_ens = nse_single(targets_single.reshape(-1), preds_ensemble.reshape(-1))
    kge_ens = kge_single(targets_single.reshape(-1), preds_ensemble.reshape(-1))
    rmse_ens = float(np.sqrt(np.mean((targets_single - preds_ensemble) ** 2)))
    mae_ens = float(np.mean(np.abs(targets_single - preds_ensemble)))

    ensemble_results[rid] = {
        "reservoir": info["name"], "rid": rid,
        "nse_ensemble": round(nse_ens, 4),
        "kge_ensemble": round(kge_ens, 4) if not np.isnan(kge_ens) else None,
        "rmse_ensemble": round(rmse_ens, 2), "mae_ensemble": round(mae_ens, 2),
        "nse_single": round(nse_single_val, 4), "nse_basin": round(nse_basin_val, 4),
        "improved_over_single": bool(nse_ens > nse_single_val),
        "improved_over_basin": bool(nse_ens > nse_basin_val),
    }
    tag = "tot hon ca 2" if (ensemble_results[rid]["improved_over_single"] and ensemble_results[rid]["improved_over_basin"]) else "khong cai thien ro"
    print(
        f"[ENSEMBLE {ENSEMBLE_WEIGHT_SINGLE:.0%}/{ENSEMBLE_WEIGHT_BASIN:.0%}] {info['name']:20s} | "
        f"NSE_single={nse_single_val:.3f} NSE_basin={nse_basin_val:.3f} -> NSE_ensemble={nse_ens:.3f} ({tag})"
    )

if ensemble_results:
    ensemble_df = pd.DataFrame(list(ensemble_results.values()))
    ensemble_path = f"{OUTPUT_ROOT}/ensemble_results.xlsx"
    ensemble_df.to_excel(ensemble_path, index=False)
    print(f"\nDa luu ket qua ensemble ({len(ensemble_results)} ho du dieu kien) -> {ensemble_path}")
else:
    print("\nKhong co ho nao du dieu kien ensemble (can ca 2 file .npy va NSE gan nhau).")
